# 01 · North Carolina Flood-Event EDA

**StormRoute — Carolina Data Challenge 2026, Natural Science Track**

This notebook is the **Milestone 2 verification gate**. No production modeling,
scoring, API, or website work begins until it has been executed end to end,
reviewed by two teammates, and merged.

It must read as an analytical narrative, not a scratchpad: each section states the
question it tests, answers it with code, and closes with a short conclusion.

| | |
|---|---|
| **Run mode** | `STORMROUTE_DATA_MODE=sample` in CI (tracked fixtures) · `full` in Codespaces |
| **Execute** | `make eda` — restart kernel, run all, fail on any exception |
| **Figures** | exported to `outputs/figures/eda_*.png` with stable names |
| **Report** | findings summarized in `reports/eda/eda_findings.md` |

> **Status:** all sections implemented: Mason's (1–5, 9–12) and Cameron's (3's missingness block
> and 6–8). Awaiting a second-person review of Cameron's sections, the manual spot check, two
> reviewer approvals, and the gate decision in section 12.

## 1. Objective and scope

**Question.** What exactly are we predicting, and what are we deliberately not predicting?

A traveler decides on a route and a departure time before the trip starts. StormRoute
estimates, for every North Carolina county and six-hour window, the probability that a
reported **Flood, Flash Flood, or Debris Flow** event begins there. Route-level exposure is
built from those probabilities later.

The definitions below are read from `configs/`, not typed here, so this notebook cannot
drift away from the pipeline. This gate covers flooding only: Phase 1 of ADR 0004, which
adds other hazard families only after the flood model passes Milestone 4.

In [ ]:
import pandas as pd

from stormroute.config import load_config

DATA = load_config("data")
MODEL = load_config("model")

splits = {name: spec["years"] for name, spec in MODEL["splits"].items() if isinstance(spec, dict)}
pd.DataFrame(
    [
        (
            "Geography",
            f"{DATA['geography']['state']} (state FIPS {DATA['geography']['state_fips']}), "
            f"{DATA['geography']['expected_county_count']} counties",
        ),
        ("Period", f"{DATA['period']['start_year']}–{DATA['period']['end_year']}"),
        (
            "Prediction unit",
            f"county × {DATA['prediction_unit']['window_hours']}-hour window, "
            f"anchored at {DATA['prediction_unit']['window_anchor_utc']} UTC",
        ),
        ("Qualifying hazards", ", ".join(DATA["label"]["hazard_event_types"])),
        ("Positive label", DATA["label"]["onset_rule"]),
        ("Negative label means", DATA["label"]["negative_meaning"]),
        *((f"Split: {name}", ", ".join(map(str, years))) for name, years in splits.items()),
        ("Never a model input", ", ".join(MODEL["features"]["excluded"])),
    ],
    columns=["Definition", "Value"],
).set_index("Definition")

**Conclusion.** The scope is fixed by configuration. Injuries, deaths, and damage are
**descriptive outcomes only**: they are recorded after an event and can never be model
inputs. Sections 3–5 test whether the NOAA records support this scope, and section 9 tests
whether the label can be built.

## 2. Reproducibility information

**Question.** Could a teammate re-run this notebook tomorrow and get the same numbers?

`STORMROUTE_DATA_MODE=sample` runs on the tracked fixture (what CI does). `full` reads the
downloaded NOAA files named in the download record. Figures from a sample run go to
`outputs/figures/sample/`, which is ignored, so they can never overwrite the real ones.

In [ ]:
import hashlib
import json
import platform
import subprocess
import sys
from datetime import UTC, datetime
from importlib.metadata import version

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Markdown, display
from matplotlib.patches import Patch

from stormroute.config import REPO_ROOT, data_mode, data_path
from stormroute.data.noaa import (
    RAW_COLUMNS,
    RECORD_FILENAME,
    active_detail_files,
    clean_events,
    read_details,
)

MODE = data_mode()
SEED = MODEL["final_model"]["random_seed"]
FIGURES = (
    REPO_ROOT / "outputs" / "figures"
    if MODE == "full"
    else REPO_ROOT / "outputs" / "figures" / "sample"
)

if MODE == "sample":
    SOURCE_PATHS = [data_path("sample") / "storm_events_sample.csv"]
    sources = pd.DataFrame(
        {
            "file": [p.name for p in SOURCE_PATHS],
            "sha256": [hashlib.sha256(p.read_bytes()).hexdigest()[:16] for p in SOURCE_PATHS],
        }
    )
else:
    period = range(DATA["period"]["start_year"], DATA["period"]["end_year"] + 1)
    SOURCE_PATHS = active_detail_files(data_path("noaa_raw"), period)
    record = json.loads((data_path("noaa_raw") / RECORD_FILENAME).read_text())
    sources = pd.DataFrame(
        [
            {
                "year": record["files"][p.name]["year"],
                "file": p.name,
                "sha256": record["files"][p.name]["sha256"][:16],
                "recorded_at_utc": record["files"][p.name]["recorded_at_utc"],
            }
            for p in SOURCE_PATHS
        ]
    )

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    commit = "unavailable"

manifest = yaml.safe_load((REPO_ROOT / "data" / "data_manifest.yaml").read_text())
print(f"Analysis run (UTC): {datetime.now(UTC):%Y-%m-%d %H:%M}")
print(f"Data mode:          {MODE}")
print(f"Git commit:         {commit}")
print(
    f"Data manifest:      version {manifest['manifest_version']}, "
    f"updated {manifest['last_updated']}"
)
print(f"Python:             {sys.version.split()[0]} on {platform.system()}")
print(
    "Packages:           "
    + ", ".join(
        f"{pkg} {version(pkg)}"
        for pkg in ("stormroute", "pandas", "numpy", "matplotlib", "pyarrow")
    )
)
print(f"Random seed:        {SEED}")
print(f"Figures written to: {FIGURES.relative_to(REPO_ROOT)}")
sources

In [ ]:
# Figure standard: validated palette, recessive chrome, source note on every figure.
COLORS = {
    "surface": "#fcfcfb",
    "ink": "#0b0b0b",
    "ink_secondary": "#52514e",
    "muted": "#898781",
    "grid": "#e1e0d9",
    "baseline": "#c3c2b7",
    "accent": "#2a78d6",  # categorical slot 1: the thing the figure is about
    "context": "#898781",  # neutral for context marks
}
mpl.rcParams.update(
    {
        "figure.facecolor": COLORS["surface"],
        "axes.facecolor": COLORS["surface"],
        "savefig.facecolor": COLORS["surface"],
        "font.family": "sans-serif",
        "font.size": 11,
        "text.color": COLORS["ink"],
        "axes.labelcolor": COLORS["ink_secondary"],
        "xtick.color": COLORS["muted"],
        "ytick.color": COLORS["ink_secondary"],
        "axes.edgecolor": COLORS["baseline"],
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "grid.color": COLORS["grid"],
        "grid.linewidth": 0.8,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
    }
)
SOURCE_NOTE = "Source: NOAA NCEI Storm Events Database, details files " + (
    "(tracked sample fixture)"
    if MODE == "sample"
    else f"{DATA['period']['start_year']}–{DATA['period']['end_year']}"
)


def save_figure(fig: plt.Figure, name: str) -> None:
    FIGURES.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES / f"{name}.png", dpi=200, bbox_inches="tight")
    print(f"saved {(FIGURES / f'{name}.png').relative_to(REPO_ROOT)}")


def conclude(text: str) -> None:
    display(Markdown(f"**Conclusion.** {text}"))


def n(count: int, noun: str, plural: str | None = None) -> str:
    return f"{count:,} {noun if count == 1 else plural or noun + 's'}"


def pct(part: float, whole: float) -> str:
    if not whole or pd.isna(part):
        return "n/a"
    value = 100 * part / whole
    return f"{value:.3f}%" if 0 < value < 1 else f"{value:.1f}%"

**Conclusion.** Every number below traces to the files and checksums listed above, the
commit printed with them, and a fixed seed. Paths resolve from the repository root.

## 3. Schema and first-pass inspection

**Question.** Is the raw data shaped the way the NCEI documentation says it is?

Everything is read as **text**, exactly as published, so no county code or time loses a
leading zero to number parsing. The analysis reads a subset of the published columns.
Narratives are skipped because they are large and are never a model input.

In [ ]:
published_columns = pd.read_csv(SOURCE_PATHS[0], nrows=0).columns.tolist()
ANALYSIS_COLUMNS = [*RAW_COLUMNS, "BEGIN_DATE_TIME", "END_DATE_TIME", "BEGIN_LAT", "BEGIN_LON"]
raw = read_details(SOURCE_PATHS, ANALYSIS_COLUMNS)
raw_nc = raw[raw["STATE_FIPS"].str.zfill(2) == DATA["geography"]["state_fips"]]

print(f"Published columns: {len(published_columns)}  |  analysed: {len(ANALYSIS_COLUMNS)}")
print(f"Rows, all states:  {len(raw):,}  |  North Carolina: {len(raw_nc):,}")
print(f"Column dtypes:     {raw.dtypes.astype(str).value_counts().to_dict()} (all text by design)")

In [ ]:
raw_nc[
    [
        "EVENT_ID",
        "EVENT_TYPE",
        "CZ_TYPE",
        "CZ_FIPS",
        "CZ_NAME",
        "BEGIN_DATE_TIME",
        "END_DATE_TIME",
        "CZ_TIMEZONE",
        "DAMAGE_PROPERTY",
    ]
].sample(min(5, len(raw_nc)), random_state=SEED)

In [ ]:
duplicate_ids = int(raw["EVENT_ID"].duplicated().sum())
exact_duplicates = int(raw.duplicated().sum())
pd.DataFrame(
    [
        ("Duplicated EVENT_ID values", duplicate_ids),
        ("...of which exact duplicate rows", exact_duplicates),
        ("Distinct STATE values", raw["STATE"].nunique()),
        ("Distinct EVENT_TYPE values (all states)", raw["EVENT_TYPE"].nunique()),
        ("Distinct EVENT_TYPE values (NC)", raw_nc["EVENT_TYPE"].nunique()),
    ],
    columns=["Check", "Value"],
).set_index("Check")

In [ ]:
pd.crosstab(raw_nc["CZ_TYPE"], "NC rows").rename(
    index={"C": "C: county", "Z": "Z: forecast zone", "M": "M: marine"}
)

### Missingness in the North Carolina source rows

**Question.** Which source fields are blank before hazard filtering and cleaning?

Count null, empty, and whitespace-only values across all NC source rows, including
repeated records and non-target hazards. Each field uses the same row denominator.
A literal zero is observed data, not missing. Invalid nonblank values are a separate
validation problem. Coordinate availability can depend on event type; this table
alone cannot establish whether a field is suitable for modeling.


In [ ]:
missing_mask = raw_nc.apply(lambda column: column.isna() | column.str.strip().eq(""))
missingness = pd.DataFrame(
    {
        "field": raw_nc.columns,
        "rows": len(raw_nc),
        "missing_count": missing_mask.sum().to_numpy(dtype=int),
    }
)
missingness["missing_fraction"] = (
    missingness["missing_count"] / len(raw_nc) if len(raw_nc) else np.nan
)
missingness = missingness.sort_values(
    ["missing_count", "field"], ascending=[False, True], ignore_index=True
)
assert missingness["missing_count"].between(0, len(raw_nc)).all()
display(missingness)

metrics_dir = REPO_ROOT / "outputs" / "metrics"
if MODE == "sample":
    metrics_dir = metrics_dir / "sample"
metrics_dir.mkdir(parents=True, exist_ok=True)
missingness_payload = {
    "data_mode": MODE,
    "population": "North Carolina source rows before hazard filtering and deduplication",
    "missing_definition": "null, empty, or whitespace-only; literal zero is not missing",
    "source_files": [p.name for p in SOURCE_PATHS],
    "source_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in SOURCE_PATHS},
    "git_commit": commit,
    "rows": len(raw_nc),
    "fields": json.loads(missingness.to_json(orient="records")),
}
metrics_path = metrics_dir / "eda_missingness.json"
metrics_path.write_text(
    json.dumps(missingness_payload, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print(f"Saved {metrics_path.relative_to(REPO_ROOT)}")

fig, ax = plt.subplots(figsize=(10, max(5, len(missingness) * 0.25)))
ax.barh(
    missingness["field"], missingness["missing_fraction"].fillna(0) * 100, color=COLORS["accent"]
)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Missing source rows (%)")
ax.set_ylabel("Source field")
ax.set_title(f"Missing fields in NC source rows ({MODE} mode; n={len(raw_nc):,})")
ax.xaxis.grid(True)
ax.set_axisbelow(True)
if not len(raw_nc):
    ax.text(0.5, 0.5, "No NC rows: percentages undefined", transform=ax.transAxes, ha="center")
fig.text(0.01, -0.02, SOURCE_NOTE + ". Before hazard filtering and deduplication.", fontsize=9)
fig.tight_layout()
save_figure(fig, "eda_missingness")
display(fig)
plt.close(fig)

fields_with_missing = int((missingness["missing_count"] > 0).sum())
conclude(
    f"{fields_with_missing} of {len(missingness)} inspected fields contain blanks across "
    f"{len(raw_nc):,} NC source rows. "
    + ("No NC rows were available; missing fractions are undefined. " if not len(raw_nc) else "")
    + (
        "These synthetic fixture results are a smoke test, not NC population estimates. "
        if MODE == "sample"
        else ""
    )
    + "Missingness does not measure invalid nonblank values or prove predictor suitability."
)

In [ ]:
if duplicate_ids == 0:
    identity = "every EVENT_ID is unique"
elif duplicate_ids == exact_duplicates:
    identity = (
        f"{n(duplicate_ids, 'EVENT_ID')} repeat, and every repeat is an exact duplicate row, "
        "which ingestion removes"
    )
else:
    identity = (
        f"{n(duplicate_ids - exact_duplicates, 'EVENT_ID')} repeat with **different** content, "
        "which ingestion rejects"
    )
conclude(
    f"Across {n(len(SOURCE_PATHS), 'source file')} there are {len(raw):,} rows, "
    f"{len(raw_nc):,} of them in North Carolina, and {identity}. NCEI publishes county "
    "codes, days, and times **unpadded** (e.g. `21` for Buncombe), so ingestion pads every "
    "one explicitly."
)

## 4. Time-field validation

**Question.** Can every event be placed on one unambiguous UTC timeline?

**Conversion rule.** NCEI splits each timestamp into `YEARMONTH`, `DAY`, and `TIME`, all
unpadded, and records a fixed offset in `CZ_TIMEZONE` (e.g. `EST-5`). Ingestion applies that
offset as written, with no daylight-saving lookup: `UTC = local − offset`.

The timestamps are then checked against NCEI's own `BEGIN_DATE_TIME` and `END_DATE_TIME`
text fields, an independent rendering of the same instant.

In [ ]:
events, report = clean_events(raw, source_files=[p.name for p in SOURCE_PATHS])
pd.DataFrame(
    [
        ("Events after all filters", report.events_out),
        ("Excluded: begin time invalid", report.excluded_invalid_begin),
        ("Excluded: end time invalid", report.excluded_invalid_end),
        ("Excluded: end before begin", report.excluded_end_before_begin),
        ("Kept: end time missing (labels need only begin)", report.missing_end_kept),
        ("Kept: zero duration (end = begin)", report.zero_duration),
    ],
    columns=["Timestamp check", "Events"],
).set_index("Timestamp check")

In [ ]:
published = (
    raw_nc.drop_duplicates("EVENT_ID")
    .set_index("EVENT_ID")
    .loc[events["event_id"], ["BEGIN_DATE_TIME", "END_DATE_TIME", "CZ_TIMEZONE"]]
)
offset = pd.to_timedelta(published["CZ_TIMEZONE"].str.extract(r"-(\d+)$")[0].astype(int), unit="h")


def as_local(stamps: pd.Series) -> pd.Series:
    return pd.to_datetime(stamps.replace("", None), format="%d-%b-%y %H:%M:%S")


begin_local = as_local(published["BEGIN_DATE_TIME"]).to_numpy()
end_local = as_local(published["END_DATE_TIME"]).to_numpy()
ours_begin = (events["begin_utc"].dt.tz_localize(None) - offset.to_numpy()).to_numpy()
ours_end = (events["end_utc"].dt.tz_localize(None) - offset.to_numpy()).to_numpy()

begin_mismatch = int((begin_local != ours_begin).sum())
end_both = ~pd.isna(end_local) & ~pd.isna(ours_end)
end_mismatch = int((end_local[end_both] != ours_end[end_both]).sum())
print(f"Begin times disagreeing with BEGIN_DATE_TIME: {begin_mismatch} of {len(events):,}")
print(f"End times disagreeing with END_DATE_TIME:     {end_mismatch} of {int(end_both.sum()):,}")
print(f"CZ_TIMEZONE values on NC rows: {raw_nc['CZ_TIMEZONE'].value_counts().to_dict()}")

In [ ]:
local_begin = pd.Series(begin_local, index=events.index)
local_end = pd.Series(end_local, index=events.index)
on_boundary = (events["begin_utc"].dt.minute == 0) & (events["begin_utc"].dt.hour % 6 == 0)
CASES = {
    "Crosses local midnight": (local_end.dt.date != local_begin.dt.date) & local_end.notna(),
    "Crosses UTC midnight": (events["end_utc"].dt.date != events["begin_utc"].dt.date)
    & events["end_utc"].notna(),
    "Crosses a UTC year boundary": events["end_utc"].dt.year > events["begin_utc"].dt.year,
    "Begins exactly on a six-hour UTC boundary": on_boundary,
    "Zero duration on a six-hour UTC boundary": on_boundary
    & (events["end_utc"] == events["begin_utc"]),
}
pd.DataFrame({"Events": {case: int(mask.sum()) for case, mask in CASES.items()}}).rename_axis(
    "Case"
)

### Standard time or clock time?

Every North Carolina record says `EST-5`, including summer events. The rule behind that
comes from [NWS Instruction 10-1605](https://www.weather.gov/media/directives/010_pdfs/pd01016005curr.pdf),
§2.3, which governs how Storm Data is prepared: *"Use local standard time in 24-hour clock
throughout the year."* The 2007 version of the directive says the same, so the rule covers
all of 2015–2024. The fixed `EST-5` conversion is therefore **correct by policy**.

What policy cannot guarantee is that every preparer complied. If a summer event were entered
in clock time (EDT), it would be one hour late in UTC. Few narratives quote a zoned time,
too few to test compliance statistically. The size of the exposure is measured here, and
the manual spot check verifies compliance.

In [ ]:
in_dst = (
    events["begin_utc"].dt.tz_convert("America/New_York").map(lambda t: t.dst() != pd.Timedelta(0))
)
print(
    f"Events beginning while Eastern daylight time is in force: {int(in_dst.sum()):,} "
    f"of {len(events):,} ({pct(in_dst.sum(), len(events))})"
)

### Spot-check candidates

The gate requires at least ten events checked **by a person** against NOAA's event pages and
NWS products. These are chosen deterministically to cover the riskiest cases, with three
daylight-saving-period events to verify standard-time compliance. In full mode they are written to
`reports/eda/spot_check_candidates.csv`, with blank columns for the reviewer.

In [ ]:
durations = events["end_utc"] - events["begin_utc"]
helene = events["begin_utc"].between(
    pd.Timestamp("2024-09-25", tz="UTC"), pd.Timestamp("2024-09-29", tz="UTC")
)
strata = [
    ("Daylight-saving period (EST-5 compliance)", in_dst, 3),
    ("Standard-time period", ~in_dst, 1),
    ("Zero duration", durations == pd.Timedelta(0), 1),
    ("Crosses local midnight", CASES["Crosses local midnight"], 1),
    ("Debris Flow", events["event_type"] == "Debris Flow", 1),
    ("Helene period", helene, 1),
]
chosen: list[tuple[str, str]] = []
for label, mask, size in strata:
    pool = events[mask & ~events["event_id"].isin([e for e, _ in chosen])]
    chosen += [(e, label) for e in pool.sample(min(size, len(pool)), random_state=SEED)["event_id"]]
remaining = events[~events["event_id"].isin([e for e, _ in chosen])]
if len(remaining):
    chosen.append(
        (remaining.loc[durations[remaining.index].idxmax(), "event_id"], "Longest duration")
    )
remaining = events[~events["event_id"].isin([e for e, _ in chosen])]
fill = max(0, 10 - len(chosen))
chosen += [
    (e, "Random")
    for e in remaining.sample(min(fill, len(remaining)), random_state=SEED)["event_id"]
]

picked = events.set_index("event_id").loc[[e for e, _ in chosen]]
spot_check = pd.DataFrame(
    {
        "event_id": picked.index,
        "why_selected": [label for _, label in chosen],
        "event_type": picked["event_type"].astype(str).to_numpy(),
        "county_fips": picked["county_fips"].to_numpy(),
        "begin_as_published": published.loc[picked.index, "BEGIN_DATE_TIME"].to_numpy(),
        "begin_utc": picked["begin_utc"].dt.strftime("%Y-%m-%d %H:%M").to_numpy(),
        "ncei_event_page": [
            f"https://www.ncdc.noaa.gov/stormevents/eventdetails.jsp?id={e}" for e in picked.index
        ],
        "reviewer": "",
        "county_matches": "",
        "utc_time_matches_nws_product": "",
        "notes": "",
    }
)
if MODE == "full":
    target = REPO_ROOT / "reports" / "eda" / "spot_check_candidates.csv"
    spot_check.to_csv(target, index=False)
    print(f"written {target.relative_to(REPO_ROOT)}")
spot_check.iloc[:, :6]

In [ ]:
bad_end = report.excluded_invalid_end + report.excluded_end_before_begin
checked_ends = int(end_both.sum())
zero_on_boundary = int(CASES["Zero duration on a six-hour UTC boundary"].sum())
dst_events = int(in_dst.sum())
conclude(
    f"All {len(events):,} qualifying events have a valid begin time; "
    f"{n(bad_end, 'event')} had to be excluded for a bad end time. "
    "Ingestion's UTC timestamps agree with NCEI's own date-time text for "
    f"{len(events) - begin_mismatch:,} of {len(events):,} begin times and "
    f"{checked_ends - end_mismatch:,} of {checked_ends:,} end times. "
    f"{n(report.zero_duration, 'event')} ({pct(report.zero_duration, len(events))}) "
    f"have zero duration, and {zero_on_boundary:,} of them begin exactly on a window boundary: "
    "the onset rule places each in exactly one window. NWS policy requires standard time "
    "year-round, so the `EST-5` conversion is correct wherever preparers complied; the spot "
    f"check verifies that for the {n(dst_events, 'daylight-saving-period event')} "
    f"({pct(dst_events, len(events))})."
)

## 5. Hazard and geography filters

**Question.** How many records survive each filter, what does the hazard filter leave out,
and can every event be placed in a county?

The filters run in `stormroute.data.noaa.clean_events`, which counts every excluded row.

In [ ]:
qualifying_rows = report.rows_qualifying_type
placeable = (
    qualifying_rows
    - report.exact_duplicates_removed
    - sum(report.excluded_unresolved_geography.values())
)
funnel = pd.DataFrame(
    [
        ("All states, all event types", len(raw)),
        (f"{DATA['geography']['state']} only", report.rows_in_state),
        ("Qualifying hazard types only", qualifying_rows),
        ("Exact duplicates removed", qualifying_rows - report.exact_duplicates_removed),
        ("County-coded (placeable in a county)", placeable),
        ("Valid timestamps: final events", report.events_out),
    ],
    columns=["Stage", "Records"],
).set_index("Stage")
funnel["Share of previous"] = [""] + [
    pct(b, a) for a, b in zip(funnel["Records"][:-1], funnel["Records"][1:], strict=True)
]
funnel

In [ ]:
qualifying_nc = raw_nc[
    raw_nc["EVENT_TYPE"].isin(DATA["label"]["hazard_event_types"])
].drop_duplicates()
geo = pd.DataFrame(
    [
        ("County-coded (CZ_TYPE = C)", int((qualifying_nc["CZ_TYPE"] == "C").sum())),
        ("Zone-coded (CZ_TYPE = Z)", int((qualifying_nc["CZ_TYPE"] == "Z").sum())),
        ("Has begin coordinates", int((qualifying_nc["BEGIN_LAT"] != "").sum())),
        ("Cannot be assigned to a county", sum(report.excluded_unresolved_geography.values())),
    ],
    columns=["Qualifying NC records", "Count"],
).set_index("Qualifying NC records")
geo

### What the hazard filter leaves out

The locked definition keeps exactly three event types. Several other water-related types
also appear in North Carolina. Grouping them as "water-related" is an analyst choice for
this display only; it does not change the filter.

In [ ]:
WATER_RELATED = [
    "Flash Flood",
    "Flood",
    "Debris Flow",
    "Heavy Rain",
    "Tropical Storm",
    "Coastal Flood",
    "Hurricane (Typhoon)",
    "Storm Surge/Tide",
    "Tropical Depression",
]
KEPT = DATA["label"]["hazard_event_types"]
DISCUSSED = {"Heavy Rain", "Coastal Flood"}  # excluded types discussed in the conclusion

nc_types = raw_nc.drop_duplicates()["EVENT_TYPE"].value_counts()
plot = pd.DataFrame(
    {"type": WATER_RELATED, "reports": [int(nc_types.get(t, 0)) for t in WATER_RELATED]}
)
plot = plot[plot["reports"] > 0].assign(kept=lambda d: d["type"].isin(KEPT))
plot = pd.concat(
    [
        plot[plot["kept"]].sort_values("reports", ascending=False),
        plot[~plot["kept"]].sort_values("reports", ascending=False),
    ]
)

fig, ax = plt.subplots(figsize=(10, 5.6))
y = np.arange(len(plot))[::-1]
ax.barh(
    y,
    plot["reports"],
    height=0.62,
    edgecolor=COLORS["surface"],
    linewidth=2,
    color=np.where(plot["kept"], COLORS["accent"], COLORS["context"]),
)
ax.set_yticks(y, plot["type"])
ax.tick_params(axis="y", length=0)
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.spines["left"].set_visible(False)
ax.set_xlim(0, plot["reports"].max() * 1.12)
ax.xaxis.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))
too_short_to_read = plot["reports"].max() * 0.03
for yi, row in zip(y, plot.itertuples(), strict=True):
    if row.kept or row.type in DISCUSSED or row.reports < too_short_to_read:
        ax.text(
            row.reports + plot["reports"].max() * 0.012,
            yi,
            f"{row.reports:,}",
            va="center",
            color=COLORS["ink_secondary"],
            fontsize=10,
        )
ax.set_xlabel(f"Reports in North Carolina, {SOURCE_NOTE.split('files ')[-1]} (count)")
ax.set_title(
    f"The hazard filter keeps {int(plot['kept'].sum())} of {len(plot)} water-related event types",
    pad=28,
)
ax.text(
    0,
    1.02,
    "NOAA Storm Events reports by type; the label uses only the kept types",
    transform=ax.transAxes,
    color=COLORS["ink_secondary"],
    fontsize=11,
)
ax.legend(
    handles=[
        Patch(color=COLORS["accent"], label="Kept: qualifying hazard"),
        Patch(color=COLORS["context"], label="Excluded"),
    ],
    loc="lower right",
    frameon=False,
)
fig.text(
    0.01,
    -0.02,
    SOURCE_NOTE + ". Water-related grouping is for display only.",
    color=COLORS["muted"],
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "eda_event_type_counts")
display(fig)
plt.close(fig)

In [ ]:
excluded = plot[~plot["kept"]].set_index("type")["reports"]
unresolved = sum(report.excluded_unresolved_geography.values())
geography = (
    "is county-coded, so **no** event had to be dropped for unplaceable geography, and a "
    "zone-to-county crosswalk is unnecessary for this label."
    if unresolved == 0
    else f"except {n(unresolved, 'zone-coded record')} is county-coded; "
    "those are excluded and counted."
)
conclude(
    f"{report.events_out:,} of {len(raw):,} records survive ({pct(report.events_out, len(raw))}). "
    f"Every qualifying record {geography} "
    f"The filter leaves out {int(excluded.sum()):,} water-related reports, notably Heavy Rain "
    f"({int(excluded.get('Heavy Rain', 0)):,}) and Coastal Flood "
    f"({int(excluded.get('Coastal Flood', 0)):,}). Heavy Rain is excluded by ADR 0000. "
    "Coastal Flood is excluded by team decision: tides and surge drive it, which the rainfall "
    "and soil features cannot explain, and NWS reports it by zone, not county. Official "
    "Coastal Flood Warnings still raise trip risk through the alert floor."
)

## 6. Temporal coverage

**Question.** How do retained report counts, durations, and coverage vary?

Full-data and sample execution of this section is complete. Findings and
limitations are recorded in `reports/eda/temporal_counts_review.md`.
Counts describe reports, not hazard probabilities or reporting completeness.
Synthetic sample results remain smoke tests, not historical findings.


### Event counts (Cameron)

**Question.** How many retained qualifying reports begin in each year, month,
county, and hazard category?

Use the cleaned, deduplicated `events` table from section 4. Count each event ID
once per breakdown, using its UTC beginning time. Restrict beginnings to the
configured 2015-2024 UTC period and report exclusions explicitly: a source-year
file can contain a local December event whose UTC beginning falls in January.
Months are pooled across years, not monthly rates. Zero-filled year and month
bins mean no retained reports in these inputs, not proof of complete coverage.
County totals include observed FIPS only; identifying all zero-report counties
requires the county-boundary inventory and remains a separate task.


In [ ]:
start_year = DATA["period"]["start_year"]
end_year = DATA["period"]["end_year"]
period_start = pd.Timestamp(year=start_year, month=1, day=1, tz="UTC")
period_end = pd.Timestamp(year=end_year + 1, month=1, day=1, tz="UTC")
in_period = events["begin_utc"].ge(period_start) & events["begin_utc"].lt(period_end)
counted_events = events.loc[in_period].copy()
assert counted_events["event_id"].is_unique

annual_counts = (
    counted_events["begin_utc"]
    .dt.year.value_counts()
    .reindex(range(start_year, end_year + 1), fill_value=0)
    .sort_index()
    .rename_axis("year")
    .rename("event_count")
    .reset_index()
)
monthly_counts = (
    counted_events["begin_utc"]
    .dt.month.value_counts()
    .reindex(range(1, 13), fill_value=0)
    .sort_index()
    .rename_axis("month")
    .rename("event_count")
    .reset_index()
)
county_counts = (
    counted_events.groupby("county_fips", observed=True)
    .size()
    .rename("event_count")
    .reset_index()
    .sort_values(["event_count", "county_fips"], ascending=[False, True], ignore_index=True)
)
hazard_counts = (
    counted_events["event_type"]
    .value_counts()
    .reindex(DATA["label"]["hazard_event_types"], fill_value=0)
    .rename_axis("event_type")
    .rename("event_count")
    .reset_index()
)
count_tables = {
    "year": annual_counts,
    "month": monthly_counts,
    "county": county_counts,
    "hazard": hazard_counts,
}
for dimension, table in count_tables.items():
    assert int(table["event_count"].sum()) == len(counted_events)
    display(Markdown(f"**Counts by {dimension}**"))
    display(table)

count_payload = {
    "data_mode": MODE,
    "population": "Unique cleaned NC qualifying events with UTC beginnings in configured period",
    "time_basis": "UTC beginning; months pooled across years",
    "period_start_inclusive": period_start.isoformat(),
    "period_end_exclusive": period_end.isoformat(),
    "cleaned_events": len(events),
    "events_counted": len(counted_events),
    "excluded_outside_utc_period": int((~in_period).sum()),
    "county_coverage": "Observed county FIPS only; no complete county inventory inferred",
    "source_files": [p.name for p in SOURCE_PATHS],
    "source_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in SOURCE_PATHS},
    "git_commit": commit,
    "counts": {
        name: json.loads(table.to_json(orient="records")) for name, table in count_tables.items()
    },
}
count_path = metrics_dir / "eda_event_counts.json"
count_path.write_text(json.dumps(count_payload, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(f"Saved {count_path.relative_to(REPO_ROOT)}")

In [ ]:
for table, column, title, xlabel, filename in [
    (
        annual_counts,
        "year",
        "Qualifying event reports by UTC start year",
        "UTC start year",
        "eda_events_by_year",
    ),
    (
        monthly_counts,
        "month",
        "Qualifying event reports by UTC start month",
        "UTC start month (pooled across years)",
        "eda_monthly_seasonality",
    ),
]:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(table[column], table["event_count"], color=COLORS["accent"])
    ax.set_xticks(table[column])
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Retained unique event reports (count)")
    ax.set_title(f"{title} ({MODE} mode)")
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
    fig.text(
        0.01,
        -0.04,
        SOURCE_NOTE + ". Counts are not hazard probabilities or coverage estimates.",
        fontsize=9,
    )
    fig.tight_layout()
    save_figure(fig, filename)
    display(fig)
    plt.close(fig)

conclude(
    f"All four breakdowns reconcile to {len(counted_events):,} retained event reports; "
    f"{int((~in_period).sum()):,} cleaned events fall outside the UTC analysis period. "
    f"{len(county_counts)} county FIPS appear in these inputs. "
    + (
        "Synthetic sample counts cannot establish seasonality, reporting trends, or county risk. "
        if MODE == "sample"
        else ""
    )
    + "These are report counts, not positive county-window rates. Reporting completeness, "
    "county-size effects, and causal explanations are not established by these counts."
)
if MODE == "full":
    peak_year = annual_counts.loc[annual_counts["event_count"].idxmax()]
    peak_month = monthly_counts.loc[monthly_counts["event_count"].idxmax()]
    conclude(
        f"The largest annual count is {int(peak_year['event_count']):,} in "
        f"{int(peak_year['year'])}; the largest pooled monthly count is "
        f"{int(peak_month['event_count']):,} in month {int(peak_month['month'])}. "
        "Pooled months are not adjusted for days of exposure. A few large storms "
        "may contribute multiple county records; these descriptive totals do not "
        "establish a trend or independent storm frequency."
    )

### Independent split and episode check (Cameron)

Reconstruct onset bins from the published date-time text for the retained event IDs,
independently of section 9's grid membership calculation. This checks binning and
split counts, not the correctness of source reports or ingestion exclusions. NC
retained records must use EST-5; add five hours to obtain UTC. Count distinct
county/six-hour bins, not events. Denominators include every county and calendar
window, including windows with no report. A negative means no qualifying report.

Episode IDs are post-event grouping metadata, never predictors. One named storm
may span several episodes; the largest episode's share is not a named-storm share.
Different episodes can share a positive bin. Counts measure dependence, not the
effective number of independent observations. Test-year summaries are descriptive
only and cannot select features, thresholds, or models.


In [ ]:
review_source = (
    raw_nc.drop_duplicates("EVENT_ID")
    .set_index("EVENT_ID")
    .loc[counted_events["event_id"].tolist()]
)
assert review_source["CZ_TIMEZONE"].eq("EST-5").all()
review_onsets = pd.to_datetime(
    review_source["BEGIN_DATE_TIME"], format="%d-%b-%y %H:%M:%S", utc=True
) + pd.Timedelta(hours=5)
review_hours = int(DATA["prediction_unit"]["window_hours"])
review_bins = pd.DataFrame(
    {
        "county_fips": review_source["STATE_FIPS"].str.zfill(2)
        + review_source["CZ_FIPS"].str.zfill(3),
        "window": review_onsets.dt.normalize()
        + pd.to_timedelta((review_onsets.dt.hour // review_hours) * review_hours, unit="h"),
        "episode_id": review_source["EPISODE_ID"].str.strip(),
    }
)
review_bins["year"] = review_bins["window"].dt.year
review_positive_bins = review_bins.drop_duplicates(["county_fips", "window"])
review_rows = []
for split, spec in MODEL["splits"].items():
    if not isinstance(spec, dict):
        continue
    years = spec["years"]
    denominator = sum(
        (pd.Timestamp(year=y + 1, month=1, day=1) - pd.Timestamp(year=y, month=1, day=1)).days
        * (24 // review_hours)
        * DATA["geography"]["expected_county_count"]
        for y in years
    )
    positives = int(review_positive_bins["year"].isin(years).sum())
    split_reports = review_bins.loc[review_bins["year"].isin(years)]
    review_rows.append(
        {
            "split": split,
            "windows": denominator,
            "positives": positives,
            "negatives": denominator - positives,
            "prevalence": positives / denominator,
            "reported_episodes": int(split_reports["episode_id"].replace("", pd.NA).nunique()),
            "recall_step": 1 / positives if positives else None,
        }
    )
review_split_counts = pd.DataFrame(review_rows).set_index("split")
display(review_split_counts)

episode_rows = []
for year in range(start_year, end_year + 1):
    year_reports = review_bins.loc[review_bins["year"].eq(year)]
    known = year_reports.loc[year_reports["episode_id"].ne("")]
    sizes = known.groupby("episode_id").size().sort_values(ascending=False)
    largest = sizes.index[0] if len(sizes) else None
    largest_rows = known.loc[known["episode_id"].eq(largest)]
    positive_count = int(review_positive_bins["year"].eq(year).sum())
    largest_bins = len(largest_rows.drop_duplicates(["county_fips", "window"]))
    episode_rows.append(
        {
            "year": year,
            "reports": len(year_reports),
            "positive_windows": positive_count,
            "episodes": len(sizes),
            "missing_episode_reports": len(year_reports) - len(known),
            "largest_episode_id": largest,
            "largest_episode_reports": len(largest_rows),
            "largest_episode_report_share": len(largest_rows) / len(year_reports)
            if len(year_reports)
            else None,
            "largest_episode_positive_windows": largest_bins,
            "largest_episode_positive_share": largest_bins / positive_count
            if positive_count
            else None,
        }
    )
review_episodes = pd.DataFrame(episode_rows)
display(review_episodes)
review_payload = {
    "data_mode": MODE,
    "git_commit": commit,
    "source_sha256": count_payload["source_sha256"],
    "method": "Published start text + 5h; integer-hour bins; unique county/window pairs",
    "scope": "Retained IDs only; not independent validation of source reports or exclusions",
    "splits": json.loads(review_split_counts.reset_index().to_json(orient="records")),
    "annual_episodes": json.loads(review_episodes.to_json(orient="records")),
}
(metrics_dir / "eda_split_episode_review.json").write_text(
    json.dumps(review_payload, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
conclude(
    "Nonzero positives establish estimability, not adequate statistical power. "
    "Few episodes can make window-level uncertainty misleading; assess grouped uncertainty. "
    "Calendar-year totals cannot be attributed to a single named storm. "
    + ("Synthetic fixture counts are smoke tests only." if MODE == "sample" else "")
)

### Reported event duration (Cameron)

**Question.** How long do retained reports last, and how much duration information is missing?

Use the same unique events and UTC onset period as the count tables. Duration is
`end_utc - begin_utc`, in hours, without clipping events at midnight or year end.
Missing ends remain unknown and are excluded from duration statistics. Recorded
zero durations are counted separately and included in statistics, without implying
that the physical hazard was instantaneous. Invalid or reversed timestamps were
excluded by ingestion; this distribution describes retained reports only.
Reported duration is descriptive, never a model feature or a road-closure estimate.


In [ ]:
duration_hours = (counted_events["end_utc"] - counted_events["begin_utc"]).dt.total_seconds() / 3600
known_durations = duration_hours.dropna()
assert known_durations.ge(0).all(), "Negative duration survived ingestion"
missing_duration_count = int(duration_hours.isna().sum())
zero_duration_count = int(known_durations.eq(0).sum())
positive_duration_count = int(known_durations.gt(0).sum())
assert missing_duration_count + zero_duration_count + positive_duration_count == len(counted_events)

duration_stats = {
    "min": float(known_durations.min()) if len(known_durations) else None,
    "median": float(known_durations.median()) if len(known_durations) else None,
    "p90": float(known_durations.quantile(0.90)) if len(known_durations) else None,
    "p95": float(known_durations.quantile(0.95)) if len(known_durations) else None,
    "max": float(known_durations.max()) if len(known_durations) else None,
}
# Explicit disjoint bins keep zeros visible and retain the long-duration tail.
duration_bins = pd.DataFrame(
    {
        "duration_hours": ["0", "(0, 1]", "(1, 6]", "(6, 24]", "(24, 72]", ">72"],
        "event_count": [
            zero_duration_count,
            int((known_durations.gt(0) & known_durations.le(1)).sum()),
            int((known_durations.gt(1) & known_durations.le(6)).sum()),
            int((known_durations.gt(6) & known_durations.le(24)).sum()),
            int((known_durations.gt(24) & known_durations.le(72)).sum()),
            int(known_durations.gt(72).sum()),
        ],
    }
)
assert int(duration_bins["event_count"].sum()) == len(known_durations)
display(
    pd.DataFrame(
        [
            {
                "retained_events": len(counted_events),
                "missing_end": missing_duration_count,
                "zero_duration": zero_duration_count,
                "positive_duration": positive_duration_count,
            }
        ]
    )
)
display(pd.DataFrame([duration_stats]).rename_axis("Duration summary (hours)"))
display(duration_bins)

duration_payload = {
    "data_mode": MODE,
    "population": count_payload["population"],
    "period_start_inclusive": period_start.isoformat(),
    "period_end_exclusive": period_end.isoformat(),
    "source_files": count_payload["source_files"],
    "source_sha256": count_payload["source_sha256"],
    "git_commit": commit,
    "retained_events": len(counted_events),
    "known_duration_events": len(known_durations),
    "missing_end_events": missing_duration_count,
    "zero_duration_events": zero_duration_count,
    "positive_duration_events": positive_duration_count,
    "definition": "UTC end minus UTC begin in hours; no clipping at period boundaries",
    "quantile_method": "Linear interpolation; recorded zeros included, missing ends excluded",
    "summary_hours": duration_stats,
    "bins": json.loads(duration_bins.to_json(orient="records")),
}
duration_path = metrics_dir / "eda_event_durations.json"
duration_path.write_text(
    json.dumps(duration_payload, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print(f"Saved {duration_path.relative_to(REPO_ROOT)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(duration_bins["duration_hours"], duration_bins["event_count"], color=COLORS["accent"])
ax.set_xlabel("Reported duration (hours; unequal-width categories)")
ax.set_ylabel("Retained event reports (count)")
ax.set_title(f"Reported event durations ({MODE} mode; known n={len(known_durations):,})")
ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
ax.yaxis.grid(True)
ax.set_axisbelow(True)
if not len(known_durations):
    ax.text(0.5, 0.5, "No known durations", transform=ax.transAxes, ha="center")
fig.text(
    0.01,
    -0.04,
    SOURCE_NOTE + f". Missing ends excluded: {missing_duration_count:,}; recorded zeros included.",
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "eda_event_durations")
display(fig)
plt.close(fig)

median_text = f"{duration_stats['median']:.2f} hours" if len(known_durations) else "undefined"
conclude(
    f"Of {len(counted_events):,} retained reports, {missing_duration_count:,} have unknown "
    f"duration, {zero_duration_count:,} record zero duration, and {positive_duration_count:,} "
    f"have positive duration. Median known duration is {median_text}. "
    + (
        "These are synthetic fixture results, not estimates for NC flooding. "
        if MODE == "sample"
        else ""
    )
    + "Reported intervals do not measure how long a road was flooded or closed."
)

### Reporting gaps and annual changes (Cameron)

**Question.** Where do the inputs show months without retained qualifying reports,
and which years have the largest changes in report counts?

Count unique cleaned events by UTC onset month across the entire configured period.
Show every calendar month, including zeros. A zero means no retained qualifying
reports in these inputs, not absence of flooding or proof that reporting failed.
Consecutive zero months are review candidates. Annual changes are ranked by
absolute count change, without a significance claim or an arbitrary alarm threshold.
Percent change is undefined when the previous year's count is zero.

Source-file coverage is checked separately using the downloader's annual filename
convention. Sample mode has no annual source coverage claim. Even a complete set
of source files cannot prove complete reporting within each month. The earlier
exclusion funnel must be reviewed alongside this analysis.


In [ ]:
from stormroute.data.noaa import DETAILS_FILENAME

month_index = pd.date_range(period_start, period_end, freq="MS", inclusive="left")
onset_months = counted_events["begin_utc"].dt.strftime("%Y-%m")
month_labels = month_index.strftime("%Y-%m")
calendar_counts = onset_months.value_counts().reindex(month_labels, fill_value=0)
calendar_table = pd.DataFrame(
    {
        "month_utc": month_labels,
        "event_count": calendar_counts.to_numpy(dtype=int),
    }
)
assert int(calendar_table["event_count"].sum()) == len(counted_events)

zero_runs = []
run_start = None
for i, count in enumerate(calendar_table["event_count"]):
    if count == 0 and run_start is None:
        run_start = i
    if run_start is not None and (count != 0 or i == len(calendar_table) - 1):
        run_end = i - 1 if count != 0 else i
        zero_runs.append(
            {
                "start_month_utc": str(month_labels[run_start]),
                "end_month_utc": str(month_labels[run_end]),
                "months": run_end - run_start + 1,
            }
        )
        run_start = None

annual_changes = annual_counts.copy()
annual_changes["previous_event_count"] = annual_changes["event_count"].shift(1)
annual_changes["count_change"] = (
    annual_changes["event_count"] - annual_changes["previous_event_count"]
)
annual_changes["percent_change"] = (
    100
    * annual_changes["count_change"]
    / annual_changes["previous_event_count"].where(annual_changes["previous_event_count"].gt(0))
)
ranked_changes = (
    annual_changes.dropna(subset=["count_change"])
    .assign(absolute_change=lambda table: table["count_change"].abs())
    .sort_values(["absolute_change", "year"], ascending=[False, True], ignore_index=True)
)


source_years = sorted(
    {
        int(match["year"])
        for path in SOURCE_PATHS
        if (match := DETAILS_FILENAME.fullmatch(path.name)) is not None
    }
)
missing_source_years = (
    sorted(set(range(start_year, end_year + 1)) - set(source_years)) if MODE == "full" else None
)
coverage_payload = {
    "data_mode": MODE,
    "population": count_payload["population"],
    "source_files": count_payload["source_files"],
    "source_sha256": count_payload["source_sha256"],
    "git_commit": commit,
    "source_coverage_basis": "Annual filenames; monthly reporting completeness unverified",
    "source_years": source_years if MODE == "full" else None,
    "missing_source_years": missing_source_years,
    "calendar_months": len(calendar_table),
    "zero_report_months": int(calendar_table["event_count"].eq(0).sum()),
    "zero_report_runs": zero_runs,
    "monthly_counts": json.loads(calendar_table.to_json(orient="records")),
    "annual_changes": json.loads(annual_changes.to_json(orient="records")),
    "interpretation": "Review candidates only; reporting failures not established",
}
coverage_path = metrics_dir / "eda_reporting_coverage.json"
coverage_path.write_text(
    json.dumps(coverage_payload, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print(f"Saved {coverage_path.relative_to(REPO_ROOT)}")
display(Markdown("**Consecutive months with no retained reports**"))
display(pd.DataFrame(zero_runs, columns=["start_month_utc", "end_month_utc", "months"]))
display(Markdown("**Annual changes, largest absolute count change first**"))
display(ranked_changes)
print(
    "Missing source years:",
    missing_source_years if MODE == "full" else "not assessed in sample mode",
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
positions = np.arange(len(calendar_table))
ax.plot(positions, calendar_table["event_count"], color=COLORS["accent"], label="Retained reports")
zero_positions = positions[calendar_table["event_count"].eq(0)]
ax.scatter(
    zero_positions,
    np.zeros(len(zero_positions)),
    color=COLORS["context"],
    marker="x",
    label="No retained reports",
    zorder=3,
)
ax.set_xticks(positions[::12], month_labels[::12], rotation=45, ha="right")
ax.set_xlabel("UTC onset month (January labeled each year)")
ax.set_ylabel("Retained unique reports (count)")
ax.set_title(f"Monthly reporting coverage ({MODE} mode)")
ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
ax.set_ylim(bottom=0)
ax.yaxis.grid(True)
ax.set_axisbelow(True)
ax.legend(frameon=False)
fig.text(
    0.01,
    -0.04,
    SOURCE_NOTE + ". Zero reports do not establish missing data or absence of flooding.",
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "eda_reporting_coverage")
display(fig)
plt.close(fig)

conclude(
    f"{coverage_payload['zero_report_months']} of {len(calendar_table)} calendar months have "
    f"no retained reports, in {len(zero_runs)} consecutive runs. "
    + (
        "The sparse synthetic fixture deliberately leaves gaps; these are not historical findings. "
        if MODE == "sample"
        else ""
    )
    + "Ranked annual changes identify records to investigate, not statistically established "
    "reporting changes. Review sources, exclusions, and storm context first."
)

### Helene-period context (Cameron)

**Question.** What share of retained reports begins in the selected Helene-period
window, compared with the rest of 2024 and the full study period?

For this exploratory comparison, define the window as **2024-09-25 00:00 UTC
through 2024-09-30 00:00 UTC, end exclusive** (five complete UTC days). This is an
analysis choice around the repository's September 27 demo, not an official storm
boundary or event-attribution rule. Select by UTC onset, consistently with the
count tables. An event starting before the window is not included merely because
it overlaps it; an event starting inside is counted once even if it ends later.

Compare three disjoint groups: this window, the rest of 2024, and 2015-2023.
The groups span unequal periods, so totals and shares do not estimate relative
hazard rates. Do not attribute all reports in the window to Helene. This is
descriptive EDA only; it neither evaluates a model nor selects model settings
using the held-out 2024 year.


In [ ]:
helene_start = pd.Timestamp("2024-09-25T00:00:00Z")
helene_end = pd.Timestamp("2024-09-30T00:00:00Z")
in_helene_window = counted_events["begin_utc"].ge(helene_start) & counted_events["begin_utc"].lt(
    helene_end
)
in_2024 = counted_events["begin_utc"].dt.year.eq(2024)
helene_groups = {
    "Selected Helene-period window": in_helene_window,
    "Rest of 2024": in_2024 & ~in_helene_window,
    "2015-2023": ~in_2024,
}
assert (sum(mask.astype(int) for mask in helene_groups.values()) == 1).all()
helene_comparison = pd.DataFrame(
    [
        {
            "group": label,
            "event_count": int(mask.sum()),
            "observed_counties": int(counted_events.loc[mask, "county_fips"].nunique()),
            **{
                hazard: int((mask & counted_events["event_type"].eq(hazard)).sum())
                for hazard in DATA["label"]["hazard_event_types"]
            },
        }
        for label, mask in helene_groups.items()
    ]
)
assert int(helene_comparison["event_count"].sum()) == len(counted_events)
assert (
    helene_comparison[DATA["label"]["hazard_event_types"]].sum(axis=1)
    == helene_comparison["event_count"]
).all()
window_count = int(in_helene_window.sum())
year_count = int(in_2024.sum())
window_shares = {
    "of_2024": window_count / year_count if year_count else None,
    "of_study_period": window_count / len(counted_events) if len(counted_events) else None,
}
helene_payload = {
    "data_mode": MODE,
    "population": count_payload["population"],
    "source_files": count_payload["source_files"],
    "source_sha256": count_payload["source_sha256"],
    "git_commit": commit,
    "window_start_inclusive": helene_start.isoformat(),
    "window_end_exclusive": helene_end.isoformat(),
    "selection": "UTC event onset in an exploratory date window; not causal storm attribution",
    "events_2024": year_count,
    "events_study_period": len(counted_events),
    "window_shares": window_shares,
    "groups": json.loads(helene_comparison.to_json(orient="records")),
}
helene_path = metrics_dir / "eda_helene_comparison.json"
helene_path.write_text(
    json.dumps(helene_payload, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print(f"Saved {helene_path.relative_to(REPO_ROOT)}")
display(helene_comparison)
display(
    pd.DataFrame(
        [
            {
                "denominator": "All 2024 reports",
                "reports": year_count,
                "window_share": window_shares["of_2024"],
            },
            {
                "denominator": "All study-period reports",
                "reports": len(counted_events),
                "window_share": window_shares["of_study_period"],
            },
        ]
    )
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
labels = ["Sep 25-29, 2024 UTC", "Rest of 2024", "2015-2023"]
ax.barh(
    labels,
    helene_comparison["event_count"],
    color=[COLORS["accent"], COLORS["context"], COLORS["context"]],
)
ax.invert_yaxis()
ax.set_xlabel("Retained unique event reports (count; unequal observation periods)")
ax.set_title(f"Selected Helene-period reports in context ({MODE} mode)")
ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
ax.xaxis.grid(True)
ax.set_axisbelow(True)
for label, count in zip(labels, helene_comparison["event_count"], strict=True):
    ax.annotate(str(count), (count, label), xytext=(5, 0), textcoords="offset points", va="center")
ax.set_xlim(0, max(1, int(helene_comparison["event_count"].max())) * 1.2)
fig.text(
    0.01,
    -0.04,
    SOURCE_NOTE + ". Date-window membership does not establish storm attribution.",
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "eda_helene_comparison")
display(fig)
plt.close(fig)

conclude(
    f"The selected five-day UTC window contains {window_count:,} reports: "
    f"{pct(window_count, year_count)} of 2024 and {pct(window_count, len(counted_events))} "
    "of the study-period retained reports. "
    + ("These synthetic fixture shares are not historical findings. " if MODE == "sample" else "")
    + "Other periods remain visible; report counts over unequal durations are not hazard rates "
    "and date membership alone cannot attribute an event to Helene."
)

## 7. Geographic distribution

**Question.** How do retained reports vary across all 100 North Carolina counties?

Join counts to the validated 2024 Census county inventory. Include zero-count
counties and every county-year combination. Report counts per study year and per
1,000 square kilometers of Census land area per study year. These are descriptive
reporting frequencies, not population-adjusted risk, road exposure, or hazard
probabilities. The denominator uses the configured study period, not a claim that
reporting was complete. County size, reporting access, and hazard geography can
all affect counts; normalization alone does not remove those biases.

Sample mode uses real Census boundaries simplified for display, with synthetic
events. Do not interpret its zeros as historical absence of reports. Geometry
simplification is not used to measure area: `ALAND` is Census land area in square
meters. The display fixture is unsuitable for routing or precise spatial joins.


In [ ]:
from stormroute.data.geography import load_counties

counties = load_counties(MODE)
unmatched_counties = sorted(set(counted_events["county_fips"]) - set(counties["GEOID"]))
assert not unmatched_counties, f"Event counties absent from Census: {unmatched_counties}"
years_in_study = end_year - start_year + 1
county_map = counties.merge(
    county_counts, left_on="GEOID", right_on="county_fips", how="left", validate="one_to_one"
)
county_map["event_count"] = county_map["event_count"].fillna(0).astype(int)
county_map["land_area_km2"] = pd.to_numeric(county_map["ALAND"]) / 1_000_000
assert county_map["land_area_km2"].gt(0).all()
county_map["reports_per_study_year"] = county_map["event_count"] / years_in_study
county_map["reports_per_1000_land_km2_year"] = (
    county_map["reports_per_study_year"] / county_map["land_area_km2"] * 1000
)
assert int(county_map["event_count"].sum()) == len(counted_events)
county_year_index = pd.MultiIndex.from_product(
    [counties["GEOID"], range(start_year, end_year + 1)], names=["county_fips", "year"]
)
county_year_counts = (
    counted_events.assign(year=counted_events["begin_utc"].dt.year)
    .groupby(["county_fips", "year"])
    .size()
    .reindex(county_year_index, fill_value=0)
    .rename("event_count")
    .reset_index()
)
assert int(county_year_counts["event_count"].sum()) == len(counted_events)
county_year_counts = county_year_counts.merge(
    county_map[["GEOID", "land_area_km2"]],
    left_on="county_fips",
    right_on="GEOID",
    validate="many_to_one",
).drop(columns="GEOID")
county_year_counts["reports_per_1000_land_km2"] = (
    county_year_counts["event_count"] / county_year_counts["land_area_km2"] * 1000
)
county_summary = county_map[
    [
        "GEOID",
        "NAME",
        "event_count",
        "land_area_km2",
        "reports_per_study_year",
        "reports_per_1000_land_km2_year",
    ]
].rename(columns={"GEOID": "county_fips", "NAME": "county_name"})
zero_counties = county_summary.loc[
    county_summary["event_count"].eq(0), ["county_fips", "county_name"]
]
display(county_summary.sort_values("event_count", ascending=False))
display(Markdown("**Counties with no retained reports in these inputs**"))
display(zero_counties)
boundary_record_path = (
    data_path("sample") / "nc_counties_2024.json"
    if MODE == "sample"
    else data_path("boundaries_raw") / "download_record.json"
)
geography_payload = {
    "data_mode": MODE,
    "population": count_payload["population"],
    "source_files": count_payload["source_files"],
    "source_sha256": count_payload["source_sha256"],
    "boundary_provenance": json.loads(boundary_record_path.read_text(encoding="utf-8")),
    "git_commit": commit,
    "years_in_denominator": years_in_study,
    "counties": json.loads(county_summary.to_json(orient="records")),
    "county_years": json.loads(county_year_counts.to_json(orient="records")),
    "zero_report_counties": json.loads(zero_counties.to_json(orient="records")),
    "interpretation": "Report frequencies; not road risk or proof of complete reporting",
}
geography_path = metrics_dir / "eda_geographic_distribution.json"
geography_path.write_text(
    json.dumps(geography_payload, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print(f"Saved {geography_path.relative_to(REPO_ROOT)}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
county_map.to_crs(DATA["geography"]["crs_area"]).plot(
    column="reports_per_1000_land_km2_year",
    cmap="viridis",
    linewidth=0.3,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Reports / 1,000 land km2 / study year", "shrink": 0.7},
    ax=ax,
)
ax.set_title(f"NC retained report frequency by county ({MODE} mode)")
ax.set_axis_off()
fig.text(
    0.01,
    0.01,
    SOURCE_NOTE + "\nCensus TIGER/Line 2024. Sample mode: simplified map, synthetic counts.",
    fontsize=9,
)
fig.tight_layout(rect=(0, 0.08, 1, 1))
save_figure(fig, "eda_county_choropleth")
display(fig)
plt.close(fig)
conclude(
    f"All {len(county_summary)} counties and {len(county_year_counts)} county-years included; "
    f"{len(zero_counties)} counties have no retained reports in these inputs. "
    + (
        "Synthetic sample zeros and mapped frequencies are not NC historical findings. "
        if MODE == "sample"
        else ""
    )
    + "Area normalization cannot remove reporting bias or measure road risk."
)

## 8. Descriptive impact analysis

**Question.** What impacts are recorded for retained qualifying reports?

Use the same cleaned events and UTC onset period as section 6. Summarize injuries,
deaths, and property damage overall and by hazard. Missing values remain unknown;
statistics include observed zeros and exclude unknowns. Injury/death totals combine
direct and indirect counts; if either component is missing, the combined count is
unknown under ingestion's current rule. Totals describe reported impacts, not
unique people or independently verified losses. Damage is nominal reported USD,
not inflation-adjusted; quantiles use linear interpolation.

These are **post-event descriptive outcomes, never model predictors**. The data do
not identify which casualties involved travelers, and these summaries cannot
estimate a driver's crash risk or establish whether a road was flooded. The chart
provides context for comparing exposure before travel, not a travel-risk estimate.
Full-data interpretation remains pending; sample values are synthetic.


In [ ]:
impact_fields = {
    "injuries": "reported people",
    "deaths": "reported people",
    "property_damage_usd": "nominal USD",
}
impact_rows = []
for group in ["All hazards", *DATA["label"]["hazard_event_types"]]:
    group_events = (
        counted_events
        if group == "All hazards"
        else counted_events.loc[counted_events["event_type"].eq(group)]
    )
    for field, unit in impact_fields.items():
        known = group_events[field].dropna()
        assert known.ge(0).all(), f"Negative impact value in {field}"
        impact_rows.append(
            {
                "group": group,
                "field": field,
                "unit": unit,
                "events": len(group_events),
                "known_events": len(known),
                "missing_events": int(group_events[field].isna().sum()),
                "zero_events": int(known.eq(0).sum()),
                "positive_events": int(known.gt(0).sum()),
                "reported_total": float(known.sum()) if len(known) else None,
                "median": float(known.median()) if len(known) else None,
                "p90": float(known.quantile(0.90)) if len(known) else None,
                "p95": float(known.quantile(0.95)) if len(known) else None,
                "max": float(known.max()) if len(known) else None,
            }
        )
impact_table = pd.DataFrame(impact_rows)
assert (
    impact_table["known_events"] + impact_table["missing_events"] == impact_table["events"]
).all()
assert (
    impact_table["zero_events"] + impact_table["positive_events"] == impact_table["known_events"]
).all()
display(impact_table)
impact_payload = {
    "data_mode": MODE,
    "population": count_payload["population"],
    "period_start_inclusive": period_start.isoformat(),
    "period_end_exclusive": period_end.isoformat(),
    "source_files": count_payload["source_files"],
    "source_sha256": count_payload["source_sha256"],
    "git_commit": commit,
    "statistics": "Known values only, including zeros; linear quantiles; null when none known",
    "interpretation": "Reported outcomes, not traveler-specific risk; never model predictors",
    "summary": json.loads(impact_table.to_json(orient="records")),
}
impact_path = metrics_dir / "eda_reported_impacts.json"
impact_path.write_text(
    json.dumps(impact_payload, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print(f"Saved {impact_path.relative_to(REPO_ROOT)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)
for ax, (field, _unit) in zip(axes, impact_fields.items(), strict=True):
    row = impact_table.loc[
        (impact_table["group"] == "All hazards") & impact_table["field"].eq(field)
    ].iloc[0]
    values = [int(row["zero_events"]), int(row["positive_events"]), int(row["missing_events"])]
    ax.bar(
        ["Zero", "Positive", "Unknown"],
        values,
        color=[COLORS["context"], COLORS["accent"], COLORS["baseline"]],
    )
    ax.set_title(field.replace("_", " ").capitalize())
    ax.set_xlabel("Reported outcome category")
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
axes[0].set_ylabel("Retained event reports (count)")
fig.suptitle(f"Reported impacts: context for reducing exposure ({MODE} mode)", y=1.04)
fig.text(
    0.01,
    -0.04,
    SOURCE_NOTE + ". All reported impacts; not traveler-specific. Outcomes are not predictors.",
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "eda_reported_impacts")
display(fig)
plt.close(fig)

unknown_damage = int(counted_events["property_damage_usd"].isna().sum())
conclude(
    f"Impact summaries cover {len(counted_events):,} retained event reports; "
    f"{unknown_damage:,} have unknown property damage. Observed zeros remain distinct "
    "from missing values, and known-value denominators accompany every summary. "
    + (
        "Synthetic fixture impacts cannot support claims about actual losses or casualties. "
        if MODE == "sample"
        else ""
    )
    + "These outcomes do not measure an individual traveler's risk."
)

## 9. Label construction experiment

**Question.** Can the locked label actually be built, how rare is the positive class, and
does every temporal split contain positives?

The grid is every North Carolina county crossed with every six-hour window from
2015-01-01 to 2024-12-31 UTC. A window is positive when a qualifying event **begins** in it
(`window_start <= begin < window_end`; ADR 0001). The specification's overlap rule is also
computed, so the cost of the choice is measured rather than assumed.

In [ ]:
from stormroute.data.validation import NC_COUNTY_FIPS

WINDOW = pd.Timedelta(hours=DATA["prediction_unit"]["window_hours"])
window_starts = pd.date_range(
    pd.Timestamp(f"{DATA['period']['start_year']}-01-01", tz="UTC"),
    pd.Timestamp(f"{DATA['period']['end_year'] + 1}-01-01", tz="UTC"),
    freq=WINDOW,
    inclusive="left",
)
grid = pd.MultiIndex.from_product(
    [sorted(NC_COUNTY_FIPS), window_starts], names=["county_fips", "window_start_utc"]
)
onset = pd.MultiIndex.from_arrays(
    [events["county_fips"].astype(str), events["begin_utc"].dt.floor(WINDOW)]
)
onset_in_grid = onset.isin(grid)
labels = pd.Series(grid.isin(onset), index=grid, name="label_flood_event")
print(f"Grid: {len(NC_COUNTY_FIPS)} counties x {len(window_starts):,} windows = {len(grid):,} rows")
print(f"Events whose onset falls outside the UTC grid: {int((~onset_in_grid).sum())}")

In [ ]:
# The specification's overlap rule, for comparison only.
first_window = events["begin_utc"].dt.floor(WINDOW)
span = (events["end_utc"].fillna(events["begin_utc"]).dt.ceil(WINDOW) - first_window) / WINDOW
exploded = events.loc[events.index.repeat(span.astype(int).clip(lower=1))]
exploded = exploded.assign(
    window_start_utc=exploded["begin_utc"].dt.floor(WINDOW)
    + WINDOW * exploded.groupby(level=0).cumcount()
)
exploded_end = exploded["end_utc"].fillna(exploded["begin_utc"])
overlapping = exploded[
    (exploded["begin_utc"] < exploded["window_start_utc"] + WINDOW)
    & (exploded_end > exploded["window_start_utc"])
]
overlap_labels = grid.isin(
    pd.MultiIndex.from_arrays(
        [overlapping["county_fips"].astype(str), overlapping["window_start_utc"]]
    )
)
events_lost_by_overlap = events.index.difference(overlapping.index).size

onset_positives = int(labels.sum())
overlap_positives = int(overlap_labels.sum())
pd.DataFrame(
    [
        ("County-windows in the grid", f"{len(grid):,}", f"{len(grid):,}"),
        ("Positive windows", f"{onset_positives:,}", f"{overlap_positives:,}"),
        ("Positive rate", pct(onset_positives, len(grid)), pct(overlap_positives, len(grid))),
        (
            "Negatives per positive",
            f"{(len(grid) - onset_positives) / max(onset_positives, 1):,.0f}",
            f"{(len(grid) - overlap_positives) / max(overlap_positives, 1):,.0f}",
        ),
        ("Events that label no window at all", "0", f"{events_lost_by_overlap:,}"),
    ],
    columns=["", "Onset rule (locked)", "Overlap rule (spec)"],
).set_index("")

### Why event rows alone cannot train a classifier

The events table holds **only positives**: every row is something that happened. A
classifier also needs the windows where nothing was reported, and those exist only in the
grid. They are the overwhelming majority of it, and a negative means *no reported event
began*, not *safe*.

In [ ]:
year_to_split = {
    year: name
    for name, spec in MODEL["splits"].items()
    if isinstance(spec, dict)
    for year in spec["years"]
}
split_order = list(dict.fromkeys(year_to_split[year] for year in sorted(year_to_split)))
frame = labels.reset_index()
frame["split"] = frame["window_start_utc"].dt.year.map(year_to_split)
by_split = (
    frame.groupby("split")["label_flood_event"]
    .agg(windows="size", positives="sum")
    .reindex(split_order)
)
by_split["positive rate"] = [
    pct(p, w) for p, w in zip(by_split["positives"], by_split["windows"], strict=True)
]
by_split["has positives"] = by_split["positives"] > 0
by_split

In [ ]:
pd.testing.assert_frame_equal(
    by_split[["windows", "positives"]].astype("int64"),
    review_split_counts[["windows", "positives"]].astype("int64"),
)
print("Independent published-text binning agrees with all split counts.")

In [ ]:
county_names = (
    raw_nc[raw_nc["CZ_TYPE"] == "C"]
    .assign(fips=lambda d: "37" + d["CZ_FIPS"].str.zfill(3))
    .drop_duplicates("fips")
    .set_index("fips")["CZ_NAME"]
    .str.title()
)
by_month = frame.groupby(frame["window_start_utc"].dt.month)["label_flood_event"].mean() * 100
by_county = (
    frame.groupby("county_fips")["label_flood_event"].mean().sort_values(ascending=False) * 100
)
zero_counties = int((by_county == 0).sum())

fig, (ax_month, ax_county) = plt.subplots(1, 2, figsize=(12, 5.2), width_ratios=[1, 1.6])
ax_month.bar(
    by_month.index,
    by_month.to_numpy(),
    width=0.7,
    color=COLORS["accent"],
    edgecolor=COLORS["surface"],
    linewidth=2,
)
ax_month.set_xticks(
    range(1, 13),
    [
        m[:1]
        for m in (
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun",
            "Jul",
            "Aug",
            "Sep",
            "Oct",
            "Nov",
            "Dec",
        )
    ],
)
ax_month.set_title("By calendar month (UTC)", fontsize=12)
ax_month.set_ylabel("Positive county-windows (%)")

ax_county.bar(range(len(by_county)), by_county.to_numpy(), width=0.8, color=COLORS["accent"])
ax_county.set_xticks([])
ax_county.set_xlabel(
    f"{len(by_county)} counties, ranked"
    + (f" ({n(zero_counties, 'county', 'counties')} with no positives)" if zero_counties else "")
)
ax_county.set_title("By county", fontsize=12)
for rank, (fips, rate) in enumerate(by_county.head(3).items()):
    ax_county.annotate(
        county_names.get(fips, fips),
        (rank, rate),
        xytext=(8, -2),
        textcoords="offset points",
        fontsize=9,
        color=COLORS["ink_secondary"],
    )
for ax in (ax_month, ax_county):
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)

fig.suptitle(
    f"Only {pct(onset_positives, len(grid))} of county-windows are positive",
    x=0.01,
    y=0.99,
    ha="left",
    fontsize=14,
    fontweight="bold",
)
fig.text(
    0.01,
    0.925,
    "Share of six-hour windows in which a Flood, Flash Flood, or Debris Flow "
    "event began, 2015–2024",
    color=COLORS["ink_secondary"],
    fontsize=11,
)
fig.text(0.01, -0.02, SOURCE_NOTE + ". Onset rule, ADR 0001.", color=COLORS["muted"], fontsize=9)
fig.tight_layout(rect=(0, 0, 1, 0.92))
save_figure(fig, "eda_class_imbalance")
display(fig)
plt.close(fig)

In [ ]:
smallest_split = by_split["positives"].idxmin()
all_splits = bool(by_split["has positives"].all())
conclude(
    f"The label is constructible: {onset_positives:,} of {len(grid):,} county-windows are "
    f"positive ({pct(onset_positives, len(grid))}), about one per "
    f"{(len(grid) - onset_positives) / max(onset_positives, 1):,.0f} negatives. "
    + (
        f"Every split has positives; the smallest is {smallest_split} "
        f"({int(by_split.loc[smallest_split, 'positives']):,}). "
        if all_splits
        else "**At least one split has no positives, which fails the gate.** "
    )
    + f"The overlap rule would add {overlap_positives - onset_positives:,} windows of ongoing "
    f"events but give {n(events_lost_by_overlap, 'event')} no window at all. "
    + (
        f"{n(int((~onset_in_grid).sum()), 'event')} begin after 2024 in UTC and fall outside "
        "the grid. "
        if (~onset_in_grid).any()
        else "Every event falls inside the UTC grid. "
    )
    + "Caveat: December 31 evening events in the 2014 file, which is not downloaded, would begin "
    "on 1 January 2015 in UTC and belong in the grid. "
    "Imbalance this severe is why PR-AUC, not accuracy or ROC-AUC, is the headline metric."
)

### Independent check: one-year experiment (Cameron)

**Question.** Can we construct the onset label, including its negative windows?

Use **2020**, a training year and leap year, selected before looking at class rates.
Build all 100 counties ? every six-hour UTC window. The locked onset rule is
`window_start <= begin < window_end`; duplicate events in one window still produce
one positive. End time is irrelevant to onset labels. Unreported windows are
negative labels, not evidence of safe roads. This is an EDA experiment, not the
production feature pipeline or a model evaluation.

For comparison only, apply the specification's literal overlap predicate
`begin < window_end AND end > window_start`. Missing ends cannot support that
comparison and are excluded and counted. This literal predicate can mark a
zero-duration report inside a window but not on its boundary. Events beginning
before the year can overlap its first window, so the comparison uses all cleaned
source events, not just events starting in 2020. Limited source coverage can still
omit carry-in events; this experiment does not prove complete historical labels.

The all-period grid above is the one the gate conclusions use, because it covers every split. The one-year experiment below repeats the construction independently for 2020, which the build guide asks for as a minimum. Its figure is saved as `eda_class_imbalance_2020.png` so it does not overwrite the all-period figure.

In [ ]:
def experiment_labels(event_table, county_ids, year):
    """Construct an EDA-only onset grid and literal overlap comparison."""
    start = pd.Timestamp(year=year, month=1, day=1, tz="UTC")
    end = pd.Timestamp(year=year + 1, month=1, day=1, tz="UTC")
    hours = DATA["prediction_unit"]["window_hours"]
    starts = pd.date_range(start, end, freq=f"{hours}h", inclusive="left")
    ends = starts + pd.Timedelta(hours=hours)
    grid = pd.MultiIndex.from_product(
        [county_ids, starts], names=["county_fips", "window_start_utc"]
    )
    onsets = event_table.loc[event_table["begin_utc"].ge(start) & event_table["begin_utc"].lt(end)]
    assert set(onsets["county_fips"]).issubset(set(county_ids))
    positive_keys = pd.MultiIndex.from_arrays(
        [onsets["county_fips"], onsets["begin_utc"].dt.floor(f"{hours}h")]
    )
    result = pd.DataFrame(index=grid)
    result["onset"] = grid.isin(positive_keys)
    overlap = np.zeros((len(county_ids), len(starts)), dtype=bool)
    county_positions = {key: i for i, key in enumerate(county_ids)}
    candidates = event_table.loc[
        event_table["end_utc"].notna()
        & event_table["begin_utc"].lt(end)
        & event_table["end_utc"].gt(start)
    ]
    for event in candidates.itertuples():
        assert event.county_fips in county_positions
        overlap[county_positions[event.county_fips]] |= (starts < event.end_utc) & (
            ends > event.begin_utc
        )
    result["overlap_comparison"] = overlap.ravel()
    return result.reset_index()

In [ ]:
EXPERIMENT_YEAR = 2020
label_grid = experiment_labels(events, counties["GEOID"].tolist(), EXPERIMENT_YEAR)
expected_windows = (
    len(counties) * (366 if pd.Timestamp(EXPERIMENT_YEAR, 1, 1).is_leap_year else 365) * 4
)
assert len(label_grid) == expected_windows
assert not label_grid.duplicated(["county_fips", "window_start_utc"]).any()
positive_windows = int(label_grid["onset"].sum())
negative_windows = len(label_grid) - positive_windows
label_grid["month"] = label_grid["window_start_utc"].dt.month
month_balance = (
    label_grid.groupby("month")["onset"].agg(windows="size", positives="sum").reset_index()
)
county_balance = (
    label_grid.groupby("county_fips")["onset"].agg(windows="size", positives="sum").reset_index()
)
for table in [month_balance, county_balance]:
    table["negatives"] = table["windows"] - table["positives"]
    table["positive_fraction"] = table["positives"] / table["windows"]
year_onsets = events.loc[events["begin_utc"].dt.year.eq(EXPERIMENT_YEAR)]
comparison = {
    "onset_positive_windows": positive_windows,
    "overlap_positive_windows": int(label_grid["overlap_comparison"].sum()),
    "onset_only_windows": int((label_grid["onset"] & ~label_grid["overlap_comparison"]).sum()),
    "overlap_only_windows": int((~label_grid["onset"] & label_grid["overlap_comparison"]).sum()),
    "year_onset_events": len(year_onsets),
    "year_onsets_missing_end": int(year_onsets["end_utc"].isna().sum()),
    "year_onsets_zero_duration": int(year_onsets["end_utc"].eq(year_onsets["begin_utc"]).sum()),
}
label_payload = {
    "data_mode": MODE,
    "experiment_year": EXPERIMENT_YEAR,
    "windows": len(label_grid),
    "counties": len(counties),
    "negative_windows": negative_windows,
    "positive_fraction": positive_windows / len(label_grid),
    "comparison": comparison,
    "source_files": count_payload["source_files"],
    "source_sha256": count_payload["source_sha256"],
    "boundary_provenance": geography_payload["boundary_provenance"],
    "git_commit": commit,
    "monthly_balance": json.loads(month_balance.to_json(orient="records")),
    "county_balance": json.loads(county_balance.to_json(orient="records")),
    "interpretation": "Onset labels from reports; negatives do not establish safe conditions",
}
label_path = metrics_dir / "eda_label_experiment.json"
label_path.write_text(json.dumps(label_payload, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(f"Saved {label_path.relative_to(REPO_ROOT)}")
display(pd.DataFrame([comparison]))
display(month_balance)
display(county_balance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(
    month_balance["month"], month_balance["positive_fraction"] * 100, color=COLORS["accent"]
)
axes[0].set_xticks(range(1, 13))
axes[0].set_xlabel("UTC month")
axes[0].set_ylabel("Positive county-windows (%)")
axes[0].set_title("Class balance by month")
axes[1].hist(county_balance["positive_fraction"] * 100, bins=10, color=COLORS["accent"])
axes[1].set_xlabel("Positive windows within county (%)")
axes[1].set_ylabel("Counties (count)")
axes[1].yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
axes[1].set_title("Distribution across all 100 counties")
fig.suptitle(f"Onset-label experiment: {EXPERIMENT_YEAR} ({MODE} mode)", y=1.04)
fig.text(
    0.01,
    -0.04,
    SOURCE_NOTE + ". Negative means no qualifying onset reported, not safe conditions.",
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "eda_class_imbalance_2020")
display(fig)
plt.close(fig)
conclude(
    f"The complete grid has {len(label_grid):,} windows: {positive_windows:,} positive and "
    f"{negative_windows:,} negative ({pct(positive_windows, len(label_grid))} positive). "
    "Event rows alone omit negative county-windows and may duplicate positive windows, "
    "so they cannot serve as the classifier's training table. "
    + (
        "This fixture-derived rate is not the NC historical class rate. "
        if MODE == "sample"
        else ""
    )
    + "The onset rule remains locked; the overlap calculation is descriptive comparison only."
)

## 10. Weather-feature feasibility check

**Question.** Are the weather features available and consistent, and can they be aligned to
event times without reading the future?

Hourly reanalysis comes from the [Open-Meteo archive API](https://open-meteo.com/en/docs/historical-weather-api)
for one **representative point** in a mountain, a Piedmont, and a coastal county. These are
city coordinates, not county centroids; centroids need the county boundaries (issue #3).
Full mode caches each response under `data/raw/weather/` (ignored). Sample mode reads a
48-hour tracked fixture.

In [ ]:
import httpx

WEATHER_POINTS = pd.DataFrame(
    [
        ("Mountain", "37021", "Buncombe", 35.5951, -82.5515),
        ("Piedmont", "37119", "Mecklenburg", 35.2271, -80.8431),
        ("Coastal", "37129", "New Hanover", 34.2257, -77.9447),
    ],
    columns=["region", "county_fips", "county", "latitude", "longitude"],
)
WEATHER_VARIABLES = [
    "precipitation",
    "temperature_2m",
    "wind_speed_10m",
    "wind_gusts_10m",
    "soil_moisture_0_to_7cm",
    "soil_moisture_28_to_100cm",
]
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_REQUESTS = {
    "era5_land": ("2024-01-01", "2024-12-31"),
    "era5_seamless": ("2014-12-29", f"{DATA['period']['end_year']}-12-31"),  # 72 h of lead-in
}


def load_open_meteo(model: str, start: str, end: str) -> tuple[pd.DataFrame, str]:
    cache = data_path("weather_raw") / "open_meteo_eda" / f"{model}_{start}_{end}.json"
    if not cache.exists():
        params = {
            "latitude": ",".join(map(str, WEATHER_POINTS["latitude"])),
            "longitude": ",".join(map(str, WEATHER_POINTS["longitude"])),
            "start_date": start,
            "end_date": end,
            "hourly": ",".join(WEATHER_VARIABLES),
            "models": model,
            "timezone": "GMT",
            "wind_speed_unit": "ms",
        }
        response = httpx.get(ARCHIVE_URL, params=params, timeout=300)
        response.raise_for_status()
        cache.parent.mkdir(parents=True, exist_ok=True)
        accessed = datetime.now(UTC).isoformat(timespec="seconds")
        cache.write_text(
            json.dumps({"accessed_at_utc": accessed, "params": params, "results": response.json()})
        )
    payload = json.loads(cache.read_text())
    frames = []
    for point, result in zip(
        WEATHER_POINTS.itertuples(index=False), payload["results"], strict=True
    ):
        hourly = pd.DataFrame(result["hourly"])
        hourly[WEATHER_VARIABLES] = hourly[WEATHER_VARIABLES].astype("float64")  # None -> NaN
        hourly.insert(0, "time_utc", pd.to_datetime(hourly.pop("time")).dt.tz_localize("UTC"))
        frames.append(
            hourly.assign(
                model=model,
                region=point.region,
                county_fips=point.county_fips,
                grid_cell=f"{result['latitude']:.2f}, {result['longitude']:.2f}",
            )
        )
    return pd.concat(frames, ignore_index=True), payload["accessed_at_utc"]


if MODE == "sample":
    weather = pd.read_csv(data_path("sample") / "weather_sample.csv", dtype={"county_fips": str})
    weather["time_utc"] = pd.to_datetime(weather["time_utc"], utc=True)
    print("Sample mode: tracked 48-hour fixture, no network access")
else:
    loaded = {model: load_open_meteo(model, *span) for model, span in WEATHER_REQUESTS.items()}
    weather = pd.concat([frame for frame, _ in loaded.values()], ignore_index=True)
    for model, (_, accessed) in loaded.items():
        print(f"{model:14s} {' to '.join(WEATHER_REQUESTS[model])}, accessed {accessed}")
weather.pivot_table(index="region", columns="model", values="grid_cell", aggfunc="first")

In [ ]:
availability = (
    weather.groupby("model")[WEATHER_VARIABLES]
    .agg(lambda s: pct(s.notna().sum(), len(s)))
    .T.rename_axis("Variable (non-missing hours)")
)
availability

In [ ]:
# Where does each era5_seamless variable come from? Compare it with era5_land hour by hour.
# dropna=False keeps era5_land's all-missing columns, which are exactly what is being checked.
both = weather.pivot_table(
    index=["region", "time_utc"], columns="model", values=WEATHER_VARIABLES, dropna=False
)
fetched_both = (
    both[("temperature_2m", "era5_land")].notna()
    & both[("temperature_2m", "era5_seamless")].notna()
)
compared = both[fetched_both]
matches_land = {
    variable: pct(
        (compared[(variable, "era5_seamless")] == compared[(variable, "era5_land")]).sum(),
        len(compared),
    )
    for variable in WEATHER_VARIABLES
}
print(f"Compared over the {len(compared):,} point-hours fetched from both models")
pd.Series(matches_land, name="era5_seamless hours identical to era5_land").to_frame()

### Leakage-safe rolling features

Open-Meteo reports each hour's precipitation as the total over the **preceding** hour, so
the value stamped at time T is already known at T. The feature for a window starting at T
sums the last *k* stamped values ending at T. It never touches an hour after T, and the
perturbation test below proves it.

In [ ]:
ROLLING_HOURS = DATA["weather"]["rolling_precip_hours"]


def trailing_features(precipitation: pd.Series) -> pd.DataFrame:
    features = {
        f"precip_{k}h": precipitation.rolling(k, min_periods=k).sum() for k in ROLLING_HOURS
    }
    features["max_precip_intensity"] = precipitation.rolling(6, min_periods=6).max()
    return pd.DataFrame(features)


seamless = weather[weather["model"] == "era5_seamless"].sort_values(["county_fips", "time_utc"])
gaps = seamless.groupby("county_fips")["time_utc"].diff().dropna()
assert (gaps == pd.Timedelta(hours=1)).all(), "hourly series has gaps; rolling sums would be wrong"

features = pd.concat(
    [
        trailing_features(group.set_index("time_utc")["precipitation"]).assign(county_fips=fips)
        for fips, group in seamless.groupby("county_fips")
    ]
).reset_index()

# Perturbation test: changing every hour after T must leave the features at T unchanged.
probe = seamless[seamless["county_fips"] == "37021"].set_index("time_utc")["precipitation"]
probe_time = probe.index[len(probe) // 2].floor(WINDOW)
perturbed = probe.where(probe.index <= probe_time, probe + 1000.0)
unchanged = (
    trailing_features(probe).loc[probe_time].equals(trailing_features(perturbed).loc[probe_time])
)
assert unchanged, "a rolling feature read an hour after the window start"
print(f"Perturbation test at {probe_time:%Y-%m-%d %H:%M} UTC: features unchanged by future hours")

In [ ]:
at_window_start = features[features["time_utc"].dt.hour % 6 == 0].rename(
    columns={"time_utc": "window_start_utc"}
)
aligned = labels.reset_index().merge(at_window_start, on=["county_fips", "window_start_utc"])
negative_p95 = aligned.loc[~aligned["label_flood_event"], "precip_24h"].quantile(0.95)
alignment = (
    aligned.groupby("label_flood_event")
    .agg(
        windows=("precip_24h", "size"),
        median_precip_24h_mm=("precip_24h", "median"),
        median_precip_72h_mm=("precip_72h", "median"),
    )
    .rename(index={True: "Positive (event began)", False: "Negative"})
)
above_p95 = aligned.loc[aligned["label_flood_event"], "precip_24h"].gt(negative_p95).mean()
print(
    f"Positive windows whose trailing 24 h rain exceeds the negatives' 95th percentile "
    f"({negative_p95:.1f} mm): {pct(above_p95, 1)}"
)
alignment.round(1)

In [ ]:
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

HELENE = (pd.Timestamp("2024-09-24", tz="UTC"), pd.Timestamp("2024-09-30", tz="UTC"))
start = max(HELENE[0], seamless["time_utc"].min())
stop = min(HELENE[1], seamless["time_utc"].max())
shown = seamless[seamless["time_utc"].between(start, stop)]
ymax = max(shown["precipitation"].max(), 1.0)

fig, axes = plt.subplots(len(WEATHER_POINTS), 1, figsize=(11, 7), sharex=True, sharey=True)
for ax, point in zip(axes, WEATHER_POINTS.itertuples(index=False), strict=True):
    rain = shown[shown["county_fips"] == point.county_fips]
    ax.bar(
        rain["time_utc"],
        rain["precipitation"],
        width=0.9 / 24,  # days
        color=COLORS["accent"],
    )
    onsets = events.loc[
        (events["county_fips"] == point.county_fips) & events["begin_utc"].between(start, stop),
        "begin_utc",
    ]
    ax.plot(
        onsets,
        [ymax * 1.12] * len(onsets),
        marker="v",
        markersize=9,
        linestyle="none",
        color="#eb6834",
        markeredgecolor=COLORS["surface"],
    )
    ax.set_ylim(0, ymax * 1.25)
    ax.set_title(
        f"{point.region}: {point.county} County ({n(len(onsets), 'reported onset')})",
        fontsize=11,
        loc="left",
    )
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)
axes[len(axes) // 2].set_ylabel("Precipitation (mm per hour)")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[-1].set_xlabel("Time (UTC)")
axes[0].legend(
    handles=[
        Patch(color=COLORS["accent"], label="Hourly precipitation"),
        Line2D(
            [],
            [],
            marker="v",
            linestyle="none",
            color="#eb6834",
            markersize=9,
            label="Reported flood onset (NOAA)",
        ),
    ],
    loc="upper right",
    frameon=False,
)
fig.suptitle(
    "Hourly rainfall and reported flood onsets during Hurricane Helene",
    x=0.01,
    y=0.99,
    ha="left",
    fontsize=14,
    fontweight="bold",
)
fig.text(
    0.01,
    0.945,
    "ERA5 precipitation via Open-Meteo (era5_seamless), one representative point per county",
    color=COLORS["ink_secondary"],
    fontsize=11,
)
fig.text(
    0.01,
    -0.01,
    "Sources: Open-Meteo archive API (ERA5, Copernicus C3S); " + SOURCE_NOTE[8:] + ".",
    color=COLORS["muted"],
    fontsize=9,
)
fig.tight_layout(rect=(0, 0, 1, 0.94))
save_figure(fig, "eda_precipitation_event_comparison")
display(fig)
plt.close(fig)

In [ ]:
land_precip = availability.loc["precipitation", "era5_land"]
median_24h = alignment["median_precip_24h_mm"]


def mm(label: str) -> str:
    value = median_24h.get(label, float("nan"))
    return "n/a" if pd.isna(value) else f"{value:.1f} mm"


conclude(
    "Weather features are available but **not from the configured source as-is**. Through "
    f"Open-Meteo, ERA5-Land has precipitation and wind for {land_precip} of hours, so they are "
    "missing entirely. The `era5_seamless` model fills them from ERA5, a coarser (~28 km) grid, "
    "while temperature and soil moisture stay on ERA5-Land's ~9 km grid. Every variable is present "
    "in `era5_seamless` for the hours fetched. Rolling features pass the perturbation test, "
    "so they "
    "read nothing after the window start. At window start, positive windows have a median trailing "
    f"24 h rainfall of {mm('Positive (event began)')} versus {mm('Negative')} for negatives; "
    "this is an alignment check, "
    "not model evidence. **Decision:** Milestone 3 fetches ERA5-Land directly from Copernicus, "
    "keeping rainfall on the ~9 km grid. ERA5-Land has no gust variable, so gusts come from ERA5. "
    "The retrieval method is ADR 0005 (proposed)."
)

## 11. Leakage audit

**Question.** Could any candidate feature carry information from after departure?

Each candidate is classified as **known before departure**, **contemporaneous** (true during
the window, so unknown at departure), or **known only after the event**. The table is then
checked against `configs/model.yaml`, so the configuration cannot quietly disagree with it.

In [ ]:
BEFORE, DURING, AFTER = "Known before departure", "Contemporaneous", "Known only after the event"
AUDIT = [
    (
        ["precip_1h", "precip_3h", "precip_6h", "precip_24h", "precip_72h"],
        "Reanalysis",
        BEFORE,
        "Keep",
        "Trailing sums ending at window start; perturbation test passes (section 10)",
    ),
    (
        ["max_precip_intensity"],
        "Reanalysis",
        BEFORE,
        "Keep",
        "Only as a trailing 6 h maximum; a maximum over the window itself would read the future",
    ),
    (
        ["temperature", "wind_speed", "wind_gust"],
        "Reanalysis",
        BEFORE,
        "Keep",
        "Value at window start",
    ),
    (
        ["soil_moisture_shallow", "soil_moisture_deep"],
        "Reanalysis",
        BEFORE,
        "Keep",
        "Value at window start; antecedent wetness",
    ),
    (["elevation_mean", "slope_mean"], "Terrain", BEFORE, "Keep", "Static"),
    (["month_sin", "month_cos"], "Calendar", BEFORE, "Keep", "Known in advance"),
    (
        ["historical_event_rate"],
        "NOAA labels",
        BEFORE,
        "Conditional",
        "Only if fit on 2015–2021; may encode reporting bias, so evaluated with and without",
    ),
    (
        ["rain during the window"],
        "Reanalysis",
        DURING,
        "Reject",
        "Unknown at departure; a live system would need a forecast (ADR 0003)",
    ),
    (
        ["centred or whole-period weather aggregates"],
        "Reanalysis",
        AFTER,
        "Reject",
        "Averages over hours after the window start",
    ),
    (["injuries", "deaths"], "NOAA", AFTER, "Reject", "Outcomes recorded after the event"),
    (["property_damage_usd", "crop damage"], "NOAA", AFTER, "Reject", "Assessed after the event"),
    (["event_narrative", "episode narrative"], "NOAA", AFTER, "Reject", "Written after the event"),
    (["end_utc", "event duration"], "NOAA", AFTER, "Reject", "Unknown until the event ends"),
    (["episode_id"], "NOAA", AFTER, "Reject", "Assigned during post-event curation"),
    (
        ["magnitude, flood cause, reporting source"],
        "NOAA",
        AFTER,
        "Reject",
        "Exist only for reported events, so they would identify positives perfectly",
    ),
    (
        ["other NOAA reports in the same window (e.g. Heavy Rain)"],
        "NOAA",
        DURING,
        "Reject",
        "Filed during or after the window",
    ),
    (
        ["Social Vulnerability Index"],
        "CDC/ATSDR",
        BEFORE,
        "Reject",
        "Policy: must never change an individual trip score (responsible AI)",
    ),
]
audit = pd.DataFrame(AUDIT, columns=["Features", "Source", "Availability", "Decision", "Reason"])
audit.assign(Features=audit["Features"].map(", ".join)).set_index("Features")

In [ ]:
decision = {name: row.Decision for row in audit.itertuples() for name in row.Features}
configured = [f for group in ("weather", "static", "temporal") for f in MODEL["features"][group]]
optional = MODEL["features"]["optional"]
excluded_by_config = list(MODEL["features"]["excluded"])

problems = (
    [
        f"{f}: configured as a feature but not audited as Keep"
        for f in configured
        if decision.get(f) != "Keep"
    ]
    + [
        f"{f}: optional feature not audited as Conditional"
        for f in optional
        if decision.get(f) != "Conditional"
    ]
    + [
        f"{f}: excluded in config but audited as {decision.get(f)}"
        for f in excluded_by_config
        if f in decision and decision[f] != "Reject"
    ]
)
unaudited_exclusions = [f for f in excluded_by_config if f not in decision]
print(f"Configured features: {len(configured)} kept, {len(optional)} optional")
print("Config agrees with the audit" if not problems else "\n".join(problems))
if unaudited_exclusions:
    print(f"Excluded in config under other names (covered above in words): {unaudited_exclusions}")

### Second, independent review (Cameron)

**Question.** Could a feature use information unavailable when a prediction is made?

The table below reviews every configured feature and explicit exclusion. Decisions
are design requirements, **not evidence that an unbuilt pipeline passes leakage
tests**. A weather value's valid time and publication/issue time are different:
trailing timestamps alone do not make retrospective ERA5-Land available live.
Reanalysis is suitable for the declared historical experiment but cannot establish
live forecast accuracy. Archived forecasts issued by the decision time are needed
for that evaluation; retrospective performance is not a proven numerical upper bound.

Weather availability, units, and temporal alignment remain unverified until section
10 is completed. Static inputs must have an available source vintage; calendar
features can be computed in advance. Historical event rates require out-of-time
construction within training and freezing before selection/calibration/test use;
publication lag and self-inclusion must be checked. Reporting bias also requires
comparison with and without that optional feature.

Post-event outcomes, target-derived future aggregates, and identifiers used for
labeling/grouping cannot become predictors. Vulnerability is evaluation/context
only and must not influence an individual trip score. This audit neither approves
the EDA gate nor substitutes for direct leakage tests in the production pipeline.

In [ ]:
# Explicit names prevent an added config feature from silently inheriting approval.
reviewed_weather = {
    "precip_1h",
    "precip_3h",
    "precip_6h",
    "precip_24h",
    "precip_72h",
    "max_precip_intensity",
    "temperature",
    "wind_speed",
    "wind_gust",
    "soil_moisture_shallow",
    "soil_moisture_deep",
}
reviewed_static = {"elevation_mean", "slope_mean"}
reviewed_temporal = {"month_sin", "month_cos"}
reviewed_optional = {"historical_event_rate"}
reviewed_groups = {
    "weather": reviewed_weather,
    "static": reviewed_static,
    "temporal": reviewed_temporal,
    "optional": reviewed_optional,
}
for group, reviewed in reviewed_groups.items():
    assert set(MODEL["features"][group]) == reviewed, f"Review changed {group} features"
configured_candidates = [name for group in reviewed_groups for name in MODEL["features"][group]]
assert len(configured_candidates) == len(set(configured_candidates))
assert not set(configured_candidates) & set(MODEL["features"]["excluded"])

audit_rows = []
for group, names in reviewed_groups.items():
    for name in sorted(names):
        if group == "weather":
            timing = "Contemporaneous valid time; reanalysis published retrospectively"
            decision = "Conditional: historical experiment only; live availability unverified"
            evidence = (
                "Verify units and availability; trailing windows only; record valid and issue times"
            )
        elif group == "static":
            timing = "Known before departure if source vintage was available"
            decision = "Conditional: verify static source vintage and county aggregation"
            evidence = (
                "Versioned source, retrieval record, aggregation QA; no outcome-derived terrain"
            )
        elif group == "temporal":
            timing = "Known before departure"
            decision = "Eligible by definition; implementation unverified"
            evidence = "Compute only from target window calendar; confirm consistent UTC convention"
        else:
            timing = "Known only after historical reports become available"
            decision = "Conditional: past-only training estimates; frozen for later splits"
            evidence = "Past-only training; train-only smoothing; check publication lag and bias"
        audit_rows.append(
            {
                "feature": name,
                "group": group,
                "timing": timing,
                "decision": decision,
                "required_evidence": evidence,
            }
        )
for name, reason in MODEL["features"]["excluded"].items():
    audit_rows.append(
        {
            "feature": name,
            "group": "excluded",
            "timing": "Known only after event or post-event curation",
            "decision": "Reject as predictor",
            "required_evidence": str(reason),
        }
    )
additional_rejections = {
    "episode_narrative": "Post-event narrative; metadata/context only",
    "event_id": "Source identifier; provenance only, not a predictor",
    "begin_utc": "Actual event onset; label construction only, not a known future event time",
    "label_flood_event": "Target only; never a predictor",
    "future_weather_aggregate": "Reject future observations; check forecast issue times",
    "future_event_rate": "Reject future or held-out labels in historical rates",
    "social_vulnerability": "Evaluation/context only; never an individual trip-score input",
}
assert not set(configured_candidates) & set(additional_rejections)
for name, reason in additional_rejections.items():
    audit_rows.append(
        {
            "feature": name,
            "group": "additional restriction",
            "timing": "Future/post-event or restricted use",
            "decision": "Reject as predictor",
            "required_evidence": reason,
        }
    )
audit_table = pd.DataFrame(audit_rows)
assert audit_table["feature"].is_unique
display(audit_table)
audit_payload = {
    "data_mode": MODE,
    "status": "Design audit only; feature availability and pipeline leakage tests pending",
    "git_commit": commit,
    "model_config_sha256": hashlib.sha256(
        (REPO_ROOT / "configs/model.yaml").read_bytes()
    ).hexdigest(),
    "data_config_sha256": hashlib.sha256(
        (REPO_ROOT / "configs/data.yaml").read_bytes()
    ).hexdigest(),
    "review": json.loads(audit_table.to_json(orient="records")),
    "live_limitation": "Reanalysis results do not establish live forecast accuracy",
}
audit_path = metrics_dir / "eda_leakage_audit.json"
audit_path.write_text(json.dumps(audit_payload, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(f"Saved {audit_path.relative_to(REPO_ROOT)}")
conclude(
    f"All {len(configured_candidates)} configured candidates are classified, along with "
    f"{len(MODEL['features']['excluded'])} configured exclusions and "
    f"{len(additional_rejections)} additional restrictions. "
    "This documents required safeguards; it does not verify weather availability, "
    "approve historical-rate construction, or prove a pipeline free of leakage."
)

In [ ]:
counts = audit["Decision"].value_counts()
conclude(
    f"{n(len(audit), 'candidate group')} audited: {int(counts.get('Keep', 0))} kept, "
    f"{int(counts.get('Conditional', 0))} conditional, {int(counts.get('Reject', 0))} rejected. "
    + (
        "The configured feature list agrees with the audit. "
        if not problems
        else f"**{n(len(problems), 'disagreement')} between config and audit.** "
    )
    + "Every kept feature is known at the window start. The remaining risk is not leakage but "
    "**train-serve skew**: features are built from reanalysis, which a live deployment would not "
    "have, so retrospective results cannot establish live forecast accuracy (ADR 0003)."
)

## 12. EDA conclusions and model decisions

The gate questions, each answered from the evidence above. The answers are computed from
earlier cells, so they cannot drift away from it.

In [ ]:
seamless_complete = seamless[WEATHER_VARIABLES].notna().mean().min()
land_missing = [
    v
    for v in WEATHER_VARIABLES
    if weather.loc[weather["model"] == "era5_land", v].notna().mean() == 0
]
split_counts = ", ".join(f"{name} {int(row.positives):,}" for name, row in by_split.iterrows())
rejected = audit.loc[audit["Decision"] == "Reject", "Features"].map(", ".join)
risks = [
    f"preparer compliance with standard time for the {pct(in_dst.sum(), len(events))} of "
    "events in daylight-saving months (policy confirmed; spot check pending)",
    "weather retrieval (ADR 0005, proposed) not yet reproducible from `make download`",
    f"reporting bias: {n(zero_counties, 'county', 'counties')} with no positives, and urban "
    f"{county_names.get(by_county.index[0], by_county.index[0])} highest",
    "weather checked at 3 representative points, not all 100 county centroids",
    "terrain features unchecked: boundaries have landed, but no elevation data is retrieved yet",
    "Cameron's sections 3 and 6–8 still need a second-person review",
]
answers = [
    (
        "Is the proposed label constructible?",
        "Yes" if onset_positives and not (~onset_in_grid).any() else "Partly",
        f"{onset_positives:,} of {len(grid):,} windows positive "
        f"({pct(onset_positives, len(grid))}); "
        "every event labels exactly one window (section 9)",
    ),
    (
        "Is the positive class large enough for the splits?",
        "Yes, thin in places" if by_split["has positives"].all() else "No",
        f"Positives by split: {split_counts} (section 9)",
    ),
    (
        "Which features are available consistently?",
        "All six in the EDA sample; Milestone 3 uses Copernicus ERA5-Land",
        f"{'All' if seamless_complete == 1 else 'At least ' + pct(seamless_complete, 1) + ' of'} "
        "hours at 3 points; ERA5-Land alone lacks "
        f"{', '.join(land_missing) or 'nothing'} (section 10)",
    ),
    (
        "Which fields are excluded, and why?",
        f"{n(len(rejected), 'candidate group')}",
        "Known only after the event, contemporaneous, or barred by policy: "
        + "; ".join(rejected)
        + " (section 11)",
    ),
    ("Which data-quality risks remain?", f"{n(len(risks), 'open risk')}", "; ".join(risks)),
    ("Does the team proceed, change scope, or stop?", "**Team decision; see below**", "—"),
]
table = "| # | Question | Answer | Evidence |\n|---|---|---|---|\n" + "\n".join(
    f"| {i} | {q} | {a} | {e} |" for i, (q, a, e) in enumerate(answers, start=1)
)
display(Markdown(table))

### Gate decision

**☐ Proceed  ☐ Change scope  ☐ Stop**: _to be recorded by Mason after both reviews._

**Draft recommendation for the team: proceed**, once these are settled:

1. **Spot check:** at least ten events checked by a person
   (`reports/eda/spot_check_candidates.csv`), including standard-time compliance.
2. **Review of Cameron's sections 3 and 6–8:** someone other than the author confirms they show no
   reporting break that would invalidate the calendar-year splits.

Decided on 2026-09-26: weather comes from **Copernicus ERA5-Land** (gusts from ERA5; retrieval
method in ADR 0005, proposed), and **Coastal Flood stays excluded**.

| Role | GitHub handle | Date |
|---|---|---|
| Decision (Product and AI lead) | _TBD_ | |
| Reviewer: interpretations and claims (Econ/Stats) | _TBD_ | |
| Reviewer: fresh-Codespace execution (CS) | _TBD_ | |